# Capítulo 16: Máquinas de Vetores de Suporte

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 9 de James et al. (2023).

> A máquina implementa, em conceito, a seguinte ideia: os vetores de entrada são levados, de forma não linear, a um espaço de características de dimensão muito alta. Nesse espaço, constrói-se uma superfície de decisão linear.
>
> — Corinna Cortes e Vladimir Vapnik, em tradução livre (1995, p. 273)

Um hiperplano divide o espaço em dois lados, e um classificador pode dar uma classe a cada lado. Quando duas classes podem ser separadas, muitos hiperplanos as separam, e a seção 16.1 mede cada um pela margem, a distância dele ao ponto de treino mais próximo, e escolhe o de maior margem. Nos 40 pontos simulados, o hiperplano de margem máxima tem margem 0,37 e se apoia em três pontos, os vetores de suporte: empurrar para longe qualquer um dos outros 37 não muda a solução. A mesma dependência deixa a fronteira frágil. Um ponto novo entre as duas nuvens gira o hiperplano 21,6 graus e leva a margem de 0,37 a 0,12. E, com as nuvens misturadas, nenhuma das 720 direções testadas separa as classes.

O classificador de vetores de suporte, da seção 16.2, aceita pontos dentro da margem e do lado errado do hiperplano: cada ponto tem uma folga $\varepsilon_i$, e a soma das folgas tem um teto, o orçamento $C$. O argumento `C` do `SVC` vai no sentido oposto ao do orçamento. Quando ele cresce de 0,01 a 10, a margem estreita de 3,14 para 0,72, os vetores de suporte caem de 40 para 21 e a soma das folgas, de 29,87 para 18,46. A inversão se explica reescrevendo o mesmo problema como perda mais penalidade, em que $\lambda = 1/(2\,\texttt{C})$: um `C` grande é um $\lambda$ pequeno, que é um orçamento pequeno. A perda dessa forma, a dobradiça, é zero para todo ponto além da margem, e por isso só os vetores de suporte decidem o hiperplano. O `C` se escolhe por validação cruzada, e com 40 pontos a escolha não é firme: 20 sementes do embaralhamento escolhem 6 valores diferentes, e o erro de teste de todos os valores da grade fica entre 0,23 e 0,26.

Uma fronteira reta não serve quando uma classe fica entre duas nuvens da outra: na seção 16.3, o classificador de vetores de suporte prevê a mesma classe para todos os pontos de teste e erra 0,25. A solução depende dos pontos só por produtos internos, e trocar o produto interno por um kernel $K(x_i, x_{i'})$ dá a máquina de vetores de suporte, que ajusta um hiperplano num espaço com mais colunas sem montá-las; no plano original, a fronteira sai curva. No kernel radial, $\gamma$ regula quanto a fronteira segue os pontos de treino: de $\gamma = 0{,}01$ a $\gamma = 100$, o erro de treino vai de 0,25 a zero, e o de teste desce até 0,1054, em $\gamma = 0{,}1$, e volta a subir até 0,2364. A validação cruzada escolhe `C` e $\gamma$ juntos, e o ajuste escolhido erra 0,1054 no teste, contra 0,0942 da regra de Bayes nos mesmos pontos.

Nos pacientes de `Heart`, o $\gamma = 0{,}1$ tem, entre os cinco modelos comparados, a maior área sob a curva ROC no treino, 0,990, e a menor no teste, 0,895. O kernel radial mede distâncias, e a escala dos preditores entra na conta: sem padronizar, o `SVC` radial com o $\gamma$ padrão erra 32 dos 90 pacientes de teste, contra 16 com o `StandardScaler`.

Com mais de duas classes, a seção 16.4 junta classificadores de duas. O um contra um ajusta um classificador por par de classes e decide por votos; o um contra todos ajusta um por classe e decide pelo maior escore. Em três nuvens simuladas, as duas abordagens discordam em 247 dos 9.000 pontos de teste e erram 0,1141 e 0,1114. O `SVC` usa sempre o um contra um; o `LinearSVC`, por padrão, o um contra todos. Na tabela `Khan`, com 2.308 genes medidos em 63 amostras de treino de quatro tipos de tumor, o `SVC` linear ajusta 6 pares, não erra nenhuma amostra de treino e erra 2 das 20 de teste. Com tantas colunas a mais que amostras, cada par se separa por um hiperplano, e de 0,01 a 100 o `C` não muda os erros.

Ao final deste capítulo, você será capaz de:

- Classificar um ponto pelo lado de um hiperplano, definir a margem e os vetores de suporte e ajustar o classificador de margem máxima com `SVC(kernel="linear", C=np.inf)`
- Ler as folgas $\varepsilon_i$ e o orçamento $C$ do classificador de vetores de suporte, converter o `C` do `scikit-learn` no $\lambda$ da forma de perda mais penalidade, $\lambda = 1/(2\,\texttt{C})$, e dizer para que lado a margem vai quando ele cresce
- Explicar por que só os vetores de suporte decidem o hiperplano, comparar a perda dobradiça com a da regressão logística e distinguir o `SVC` com `kernel="linear"` do `LinearSVC`
- Escrever $f(x)$ com um kernel, calcular à mão os kernels polinomial e radial, escolher `C` e $\gamma$ com o `GridSearchCV`, padronizar os preditores num `Pipeline` e comparar classificadores pela curva ROC do `decision_function`
- Descrever o um contra um e o um contra todos, contar os classificadores de cada um, ler as formas de `coef_` e de `decision_function` com mais de duas classes e avaliar o papel do `C` quando há muito mais colunas que observações

## Seções

| Seção | Tópico |
|---|---|
| [16.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/01-hiperplanos-e-o-classificador-de-margem-maxima.html) | Hiperplanos e o Classificador de Margem Máxima |
| [16.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/02-o-classificador-de-vetores-de-suporte.html) | O Classificador de Vetores de Suporte |
| [16.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/03-kernels.html) | Kernels |
| [16.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/04-mais-de-duas-classes.html) | Mais de Duas Classes |

## Hiperplanos e o Classificador de Margem Máxima

> **📌 Nota**
>
> Esta seção corresponde à seção 9.1 de James et al. (2023).

Duas nuvens de pontos no plano, uma de cada classe, que não se tocam: dá para traçar uma reta com uma classe de cada lado. Se uma reta separa, outras também separam, um pouco giradas ou deslocadas. Qual delas escolher? E o que os pontos de treino têm a dizer sobre essa escolha?

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("estilo-figuras.mplstyle")

Para imprimir com vírgula decimal:

In [ ]:
numero_br = lambda v, casas=2: (
    f"{v:,.{casas}f}"
    .replace(",", "_")
    .replace(".", ",")
    .replace("_", ".")
    .replace("-", "−")
)

### Um hiperplano divide o espaço

No plano, a equação $\beta_0 + \beta_1 X_1 + \beta_2 X_2 = 0$ descreve uma reta: os pontos $(X_1, X_2)$ que a satisfazem estão sobre ela. Os outros pontos do plano dão à expressão da esquerda um valor positivo ou negativo, conforme o lado da reta em que estão. A mesma ideia vale em qualquer número de dimensões.

> **🔷 Conceito**
>
> Num espaço de $p$ dimensões, um **hiperplano** é uma superfície plana de dimensão $p - 1$, que não precisa passar pela origem: os pontos $X = (X_1, \dots, X_p)$ que satisfazem
>
> $$
> \beta_0 + \beta_1 X_1 + \dots + \beta_p X_p = 0.
> $$
>
> Com $p = 2$, é uma reta; com $p = 3$, um plano. Quando a expressão da esquerda dá um valor positivo, o ponto está de um lado do hiperplano; quando dá negativo, do outro. O hiperplano divide o espaço em duas metades, e o sinal diz em qual delas o ponto está.

A reta $1 + 2X_1 + 3X_2 = 0$ é um hiperplano com $p = 2$. O valor da expressão em três pontos diz de que lado da reta cada um está:

In [ ]:
import numpy as np

def f(x1, x2):
    return 1 + 2 * x1 + 3 * x2

pontos = np.array(
    [[0, 0], [1, -1], [-1, 0.2]])
for x1, x2 in pontos:
    v = f(x1, x2)
    print(numero_br(x1, 1), ";",
        numero_br(x2, 1), "→ f =",
        numero_br(v, 1))

> **🔧 Função**
>
> - **`np.array([[0, 0], [1, -1], [-1, 0.2]])`** — um array de 3 linhas e 2 colunas, um ponto por linha.

O ponto $(0; 0)$ dá 1, e fica do lado positivo. O ponto $(1; -1)$ dá 0: está sobre a reta. O ponto $(-1; 0{,}2)$ dá $-0{,}4$, e fica do lado negativo. As duas metades do plano, com os três pontos:

In [ ]:
# Figura: O hiperplano $1 + 2X_1 + 3X_2 = 0$, a reta cinza. Na região laranja, $1 + 2X_1 + 3X_2 > 0$; na azul, $1 + 2X_1 + 3X_2 < 0$. Os losangos cinza são os três pontos da saída acima: $(0; 0)$, na região laranja; $(1; -1)$, sobre a reta; e $(-1; 0{,}2)$, na região azul.
import matplotlib.pyplot as plt

g = np.linspace(-1.5, 1.5, 201)
g1, g2 = np.meshgrid(g, g)
z = f(g1, g2)
print("z vai de",
    numero_br(z.min(), 1), "a",
    numero_br(z.max(), 1))
fig, ax = plt.subplots(
    figsize=(3.8, 3.4))
ax.contourf(g1, g2, z,
    levels=[-9, 0, 9],
    colors=["C0", "C1"],
    alpha=0.2)
ax.contour(g1, g2, z,
    levels=[0], colors="0.45",
    linewidths=1.5)
ax.scatter(pontos[:, 0],
    pontos[:, 1], s=40,
    color="0.45", marker="D",
    zorder=3)
ax.set_xticks([-1, 0, 1])
ax.set_yticks([-1, 0, 1])
ax.set_xlabel("$X_1$")
ax.set_ylabel("$X_2$")
ax.set_aspect("equal")
plt.show()

> **🔧 Função**
>
> - **`np.linspace(-1.5, 1.5, 201)`** — 201 valores igualmente espaçados de −1,5 a 1,5. **`np.meshgrid(g, g)`** — a malha de todos os pares desses valores: `g1` guarda a coordenada $X_1$ de cada nó, e `g2`, a $X_2$. Como `f` só soma e multiplica, `f(g1, g2)` calcula a expressão em todos os nós de uma vez.
> - **`z.min()`** e **`z.max()`** — o menor e o maior valor do array.
> - **`plt.subplots(figsize=(3.8, 3.4))`** — cria uma figura, `fig`, com um gráfico, `ax`, de 3,8 × 3,4 polegadas.
> - **`ax.contourf(g1, g2, z, levels, colors, alpha)`** — pinta a malha pelo valor de `z`: com `levels=[-9, 0, 9]`, o que fica entre −9 e 0 recebe a primeira cor, e o que fica entre 0 e 9, a segunda. Os valores de `z`, de −6,5 a 8,5, cabem entre −9 e 9. `alpha=0.2` deixa as cores claras.
> - **`ax.contour(g1, g2, z, levels=[0], ...)`** — desenha a curva em que `z` vale 0: aqui, a reta.
> - **`ax.scatter(x, y, s, color, marker)`** — um marcador por ponto; `marker="D"` desenha losangos, e `zorder=3` os põe por cima das regiões.
> - **`ax.set_xticks([-1, 0, 1])`** e **`ax.set_yticks(...)`** — as marcas de cada eixo. **`ax.set_xlabel(texto)`** e **`ax.set_ylabel(texto)`** — o rótulo do eixo horizontal e o do vertical.
> - **`ax.set_aspect("equal")`** — a mesma escala nos dois eixos, para as distâncias e os ângulos saírem sem distorção. **`plt.show()`** — mostra a figura.

Com $p = 3$, o hiperplano é um plano que corta o espaço em dois; com $p > 3$ não dá mais para desenhar, mas a conta é a mesma: calcular a expressão e olhar o sinal. E $p > 3$ é o caso comum: com $p$ preditores, cada observação é um ponto num espaço de $p$ dimensões, e a fronteira mais simples entre duas classes nesse espaço é um hiperplano.

### Classificar pelo lado

Agora os pontos têm classe. São $n$ observações de treino, cada uma com $p$ medidas, $x_i = (x_{i1}, \dots, x_{ip})$, e uma classe $y_i$, que vale $-1$ ou $1$. Com $f(x) = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$, um hiperplano separa as classes quando $f(x_i) > 0$ para toda observação com $y_i = 1$ e $f(x_i) < 0$ para toda observação com $y_i = -1$.

> **🔷 Conceito**
>
> Um **hiperplano separador** é um hiperplano que deixa as duas classes em lados opostos. Como $y_i$ vale $\pm 1$, as duas condições cabem numa só, que vale para todo $i = 1, \dots, n$:
>
> $$
> y_i\, f(x_i) > 0.
> $$
>
> O produto é positivo quando o sinal de $f(x_i)$ coincide com a classe. Uma observação nova $x^*$ recebe a classe do lado em que cai: $1$ se $f(x^*) > 0$, e $-1$ se $f(x^*) < 0$.

Os pontos do exemplo são simulados. Cada um tem duas coordenadas sorteadas de uma normal com média 0 e desvio padrão 1; aos da classe $1$, soma-se 2,7 nas duas coordenadas, o que afasta a nuvem deles da outra. O array `X` guarda os pontos antes do deslocamento:

In [ ]:
rng = np.random.default_rng(16)
X = rng.normal(size=(40, 2))
y = np.repeat([-1, 1], 20)
X_sep = X.copy()
X_sep[y == 1] += 2.7
for c in [-1, 1]:
    nuvem = X_sep[y == c]
    media = nuvem.mean(axis=0)
    rotulo = numero_br(c, 0)
    print(f"classe {rotulo}:",
        len(nuvem), "pontos")
    print("  média:",
        numero_br(media[0]), ";",
        numero_br(media[1]))

> **🔧 Função**
>
> - **`np.random.default_rng(16)`** — um gerador de números aleatórios com a semente 16: toda execução sorteia os mesmos números. **`rng.normal(size=(40, 2))`** — 40 linhas e 2 colunas de sorteios da normal com média 0 e desvio padrão 1.
> - **`np.repeat([-1, 1], 20)`** — cada valor repetido 20 vezes seguidas: 20 vezes −1, depois 20 vezes 1.
> - **`X.copy()`** — uma cópia de `X`; mudar a cópia não muda o original.
> - **`X_sep[y == 1] += 2.7`** — `y == 1` é `True` nas linhas da classe 1, e só essas linhas recebem 2,7 a mais, nas duas colunas.
> - **`nuvem.mean(axis=0)`** — a média de cada coluna.

São 20 pontos de cada classe, e a média da classe $1$ fica acima e à direita da média da classe $-1$ nas duas coordenadas. As duas nuvens:

In [ ]:
# Figura: Os 40 pontos simulados: a classe $-1$ em círculos azuis, a classe $1$ em triângulos laranja.
def classes(ax, pts, y):
    a = pts[y == -1]
    b = pts[y == 1]
    ax.scatter(a[:, 0], a[:, 1],
        color="C0", s=18,
        label="classe −1")
    ax.scatter(b[:, 0], b[:, 1],
        color="C1", s=24,
        marker="^",
        label="classe 1")
    ax.set_xlabel("$X_1$")
    ax.set_ylabel("$X_2$")
    ax.set_aspect("equal")

fig, ax = plt.subplots(
    figsize=(3.8, 4.2),
    layout="constrained")
classes(ax, X_sep, y)
ax.set_xlim(-3, 6)
ax.set_ylim(-3, 6)
fig.legend(ncols=2,
    loc="outside lower center")
plt.show()

> **🔧 Função**
>
> - **`ax.scatter(..., marker="^", label=...)`** — `marker="^"` desenha triângulos, e `label` é o nome do conjunto na legenda.
> - **`ax.set_xlim(-3, 6)`** e **`ax.set_ylim(-3, 6)`** — os limites do eixo horizontal e do vertical.
> - **`plt.subplots(..., layout="constrained")`** e **`fig.legend(ncols=2, loc="outside lower center")`** — a legenda da figura inteira, em 2 colunas, embaixo e fora da área dos pontos.

Uma reta que separa as nuvens deve passar pela faixa vazia entre elas. Para comparar retas, convém uma família que dependa de um número só. As retas que passam por um ponto fixo $m$, a média dos 40 pontos, e diferem só pela inclinação escrevem-se assim:

$$
\begin{aligned}
f(x) = {} & \cos\theta\,(x_1 - m_1) \\
&+ \operatorname{sen}\theta\,(x_2 - m_2).
\end{aligned}
$$

O ângulo $\theta$ gira a reta em torno de $m$. Aqui $\beta_1 = \cos\theta$ e $\beta_2 = \operatorname{sen}\theta$; abrindo os parênteses, $\beta_0 = -\beta_1 m_1 - \beta_2 m_2$, o valor que faz a reta passar por $m$. O vetor $\beta = (\cos\theta, \operatorname{sen}\theta)$ é perpendicular à reta e tem norma, isto é, magnitude, igual a 1: $\beta_1^2 + \beta_2^2 = 1$.

A expressão $f(x)$ é o produto escalar de $x - m$ com $\beta$, e a [seção 4.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/01-vetores.html) mostrou que, quando um dos vetores tem magnitude 1, o produto escalar é o comprimento da projeção do outro sobre ele. Projetar $x - m$ na direção perpendicular à reta dá a distância de $x$ até ela, então $\lvert f(x)\rvert$ é essa distância.

> **🟩 Exemplo**
>
> Com $\theta = 45^\circ$, $\cos\theta = \operatorname{sen}\theta = \sqrt{2}/2.$ O ponto $x = m + (1, 1)$ fica na direção de $\beta$, a $\sqrt{1^2 + 1^2} = \sqrt{2} \approx 1{,}41$ de $m$, e portanto a 1,41 da reta. A conta dá o mesmo:
>
> $$
> \begin{aligned}
> f(x) &= (\sqrt{2}/2)\cdot 1 + (\sqrt{2}/2)\cdot 1 \\
> &= \sqrt{2} \approx 1{,}41.
> \end{aligned}
> $$
>
> Com $\beta = (1, 1)$, que descreve a mesma reta mas tem norma $\sqrt{2}$, a conta dá $1 + 1 = 2$, e $f$ deixa de ser a distância.

Com a classe junto, $y_i f(x_i)$ é a distância com sinal: positiva quando o ponto está do lado da sua classe, negativa quando está do lado errado.

Para três ângulos, o código mostra se a reta separa as classes e qual é a menor das distâncias $y_i f(x_i)$ entre os 40 pontos:

In [ ]:
centro = X_sep.mean(axis=0)

def reta(theta, pts):
    t = np.radians(theta)
    beta = np.array(
        [np.cos(t), np.sin(t)])
    return (pts - centro) @ beta

angulos = [10, 20, 40]
print("θ: separa · menor dist.")
for theta in angulos:
    dist = y * reta(theta, X_sep)
    sep = bool(np.all(dist > 0))
    menor = numero_br(dist.min())
    print(f"{theta}°:", sep, "·",
        menor)

> **🔧 Função**
>
> - **`np.radians(theta)`** — converte o ângulo de graus para radianos, a unidade de **`np.cos`** e **`np.sin`**.
> - **`(pts - centro) @ beta`** — subtrai `centro` de cada linha de `pts` e faz, linha a linha, o produto escalar com `beta`: $\beta_1 (x_1 - m_1) + \beta_2 (x_2 - m_2)$, um valor por ponto.
> - **`np.all(dist > 0)`** — `True` quando todos os elementos de `dist` são positivos.

As três retas separam as classes, mas a menor distância muda de uma para outra, de 0,03 a 0,27. As três, sobre os pontos:

In [ ]:
# Figura: Os 40 pontos simulados, com a classe $-1$ em círculos azuis e a classe $1$ em triângulos laranja, e as três retas da saída acima, todas passando pelo centro dos pontos. As três deixam as classes em lados opostos.
fig, ax = plt.subplots(
    figsize=(3.8, 4.4),
    layout="constrained")
classes(ax, X_sep, y)
cores = ["C2", "C3", "0.45"]
for theta, cor in zip(
        angulos, cores):
    t = np.radians(theta)
    passo = np.array(
        [-np.sin(t), np.cos(t)])
    ax.axline(centro,
        centro + passo,
        color=cor, lw=1.5,
        label=f"θ = {theta}°")
ax.set_xlim(-3, 6)
ax.set_ylim(-3, 6)
fig.legend(ncols=3,
    loc="outside lower center")
plt.show()

> **🔧 Função**
>
> - **`ax.axline(p1, p2, color, lw)`** — a reta infinita que passa pelos pontos `p1` e `p2`. O vetor `passo`, $(-\operatorname{sen}\theta, \cos\theta)$, é perpendicular a $(\cos\theta, \operatorname{sen}\theta)$, então `centro + passo` é outro ponto da reta.

Se a reta passa rente a um ponto de treino, um ponto novo parecido com ele pode cair do outro lado. A distância até a reta também diz algo sobre cada ponto novo: um $x^*$ a uma distância grande da reta está longe da fronteira; um a uma distância perto de zero está quase sobre ela, e um deslocamento pequeno o passaria para o outro lado.

### A margem máxima

Se uma reta separa as classes, há infinitas outras que também separam. Para escolher uma, a medida natural é a que a saída acima já calculou: a menor distância entre a reta e os pontos de treino, a **margem**. A reta de 20° tem margem 0,27, e a de 40°, 0,03. Qual reta tem a maior margem possível?

> **🔷 Conceito**
>
> O **hiperplano de margem máxima** é o hiperplano separador mais distante das observações de treino: entre todos os que separam, o de maior margem $M$. Ele é a solução de
>
> $$
> \begin{aligned}
> &\underset{\beta_0, \dots, \beta_p,\, M}{\text{maximizar}}\ M \\
> &\text{sujeito a}\ \sum_{j=1}^{p} \beta_j^2 = 1 \\
> &\text{e}\ y_i\, f(x_i) \ge M \\
> &\text{para todo } i = 1, \dots, n.
> \end{aligned}
> $$
>
> Os coeficientes e $M$ são escolhidos juntos. A restrição $\sum_j \beta_j^2 = 1$ impede um atalho: multiplicar todos os $\beta_j$, e também $\beta_0$, por 2 descreve o mesmo hiperplano e dobra cada $y_i f(x_i)$; sem ela, $M$ cresceria sem limite. Com ela, $y_i f(x_i)$ é a distância do ponto $i$ ao hiperplano, com sinal, e a última restrição pede que todos fiquem do lado certo, a uma distância de pelo menos $M$, desde que $M > 0$. O **classificador de margem máxima** atribui a um ponto novo $x^*$ a classe indicada pelo sinal de $f(x^*)$ nesse hiperplano.

No `scikit-learn`, o classificador está no `SVC` (de *support vector classifier*, classificador de vetores de suporte), com `kernel="linear"`. O argumento `C=np.inf` não aceita nenhum ponto de treino do lado errado da margem:

In [ ]:
from sklearn.svm import SVC

modelo = SVC(kernel="linear",
    C=np.inf)
modelo.fit(X_sep, y)
print("classes_:",
    modelo.classes_)
norma = np.linalg.norm(
    modelo.coef_[0])
beta = modelo.coef_[0] / norma
b0 = modelo.intercept_[0]
beta0 = b0 / norma
M = 1 / norma
soma = (beta**2).sum()
print("β0:", numero_br(beta0))
print("β1; β2:",
    numero_br(beta[0]), ";",
    numero_br(beta[1]))
print("soma dos β² = 1:",
    bool(np.isclose(soma, 1)))
print("M:", numero_br(M))

> **🔧 Função**
>
> - **`SVC(kernel="linear", C=np.inf)`**, do módulo `sklearn.svm` — o classificador por hiperplano. Com `C=np.inf`, nenhum ponto de treino pode ficar do lado errado da margem: é o classificador de margem máxima, que só existe quando as classes se separam. A seção 16.2 mostra o que `C` faz com valores finitos.
> - **`modelo.fit(X, y)`** — ajusta o hiperplano aos pontos `X` com as classes `y`. **`modelo.classes_`** — as classes, em ordem; $f > 0$ corresponde à segunda, `classes_[1]`.
> - **`modelo.coef_`** — os coeficientes $\beta_1, \dots, \beta_p$, numa linha (`coef_[0]`); **`modelo.intercept_`** — o $\beta_0$, em `intercept_[0]`.
> - **`np.linalg.norm(v)`** — a norma do vetor, a magnitude da seção 4.1: $\sqrt{\sum_j v_j^2}$.
> - **`(beta**2).sum()`** — eleva cada elemento ao quadrado e soma. **`np.isclose(a, b)`** — `True` quando os dois números são iguais a menos do arredondamento.

Como $f > 0$ corresponde a `classes_[1]`, o lado positivo é o da classe $1$. O `SVC` guarda `coef_` e `intercept_` numa outra escala: aquela em que os pontos mais próximos do hiperplano têm $y_i f(x_i) = 1$. Dividir tudo pela norma de `coef_` leva à escala de $\sum_j \beta_j^2 = 1$, e o $1$ vira 1 dividido pela norma de `coef_`, que é $M$. O hiperplano de margem máxima é $-1{,}86 + 0{,}91\,X_1 + 0{,}42\,X_2 = 0$, com margem 0,37.

Quantos pontos você espera encontrar exatamente sobre as bordas da margem? Os pontos mais próximos do hiperplano, os que o `SVC` devolve como vetores de suporte, têm $y_i f(x_i) = 1$ na escala do `SVC` e ficam à distância $M$ na escala de norma 1:

In [ ]:
escore = modelo.decision_function(
    X_sep)
dist = y * escore / norma
sv = modelo.support_
print("support_:", sv)
print("n_support_:",
    modelo.n_support_)
print("ponto, y: y·f · distância")
for i in sv:
    yf = y[i] * escore[i]
    print(f"{i}, y = {y[i]:d}:"
        .replace("-", "−"),
        numero_br(yf), "·",
        numero_br(dist[i]))
desvio = np.abs(
    dist[sv] - M).max()
print("maior desvio de M:",
    numero_br(desvio, 4))
na_margem = np.isclose(dist, M,
    atol=1e-3)
print("à distância M (±0,001):",
    na_margem.sum())
acima = np.all(dist >= M - 1e-3)
print("nenhum mais perto:",
    bool(acima))

> **🔧 Função**
>
> - **`modelo.decision_function(X)`** — o valor de $f(x)$, na escala do `SVC`, para cada linha de `X`.
> - **`modelo.support_`** — as posições, em `X`, dos vetores de suporte. **`modelo.n_support_`** — quantos vetores de suporte há em cada classe, na ordem de `classes_`.
> - **`np.abs(v)`** — o valor absoluto de cada elemento.
> - **`np.isclose(dist, M, atol=1e-3)`** — `True` onde `dist` difere de `M` em no máximo 0,001. **`na_margem.sum()`** — num array de `True` e `False`, a soma conta os `True`.

Três pontos ficam à distância $M$: dois da classe $-1$, os pontos 2 e 19, e um da classe $1$, o ponto 26. Os três têm $y_i f(x_i) = 1{,}00$ na escala do `SVC`, e o maior desvio entre as três distâncias e $M$, 0,0001, fica bem abaixo da tolerância de 0,001 do `isclose`. Nenhum ponto fica mais perto do hiperplano. Esses três são os **vetores de suporte**: vetores porque cada observação é um ponto no espaço de $p$ dimensões, e de suporte porque o hiperplano se apoia neles.

In [ ]:
# Figura: O hiperplano de margem máxima, a reta cinza contínua, e as duas retas tracejadas, paralelas a ele e à distância $M$ de cada lado. A faixa entre as tracejadas não tem nenhum ponto. Os três pontos com anel roxo são os vetores de suporte, sobre as tracejadas. A região laranja é a que o classificador atribui à classe $1$, e a azul, à classe $-1$.
def regioes(ax, modelo):
    g = np.linspace(-3, 6, 181)
    g1, g2 = np.meshgrid(g, g)
    malha = np.column_stack(
        [g1.ravel(), g2.ravel()])
    z = modelo.decision_function(
        malha).reshape(g1.shape)
    lim = np.abs(z).max() + 1
    ax.contourf(g1, g2, z,
        levels=[-lim, 0, lim],
        colors=["C0", "C1"],
        alpha=0.2)
    estilos = ["--", "-", "--"]
    ax.contour(g1, g2, z,
        levels=[-1, 0, 1],
        colors="0.45",
        linewidths=1.5,
        linestyles=estilos)

def aneis(ax, modelo):
    sv = modelo.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1],
        s=90, facecolors="none",
        edgecolors="C3",
        linewidths=1.5,
        label="vetor de suporte")

fig, ax = plt.subplots(
    figsize=(3.8, 3.8))
regioes(ax, modelo)
classes(ax, X_sep, y)
aneis(ax, modelo)
ax.set_xlim(-3, 6)
ax.set_ylim(-3, 6)
plt.show()

> **🔧 Função**
>
> - **`np.column_stack([a, b])`** — junta os arrays como colunas; com **`g1.ravel()`** e **`g2.ravel()`**, que achatam a malha numa fila, cada linha de `malha` é um nó $(X_1, X_2)$. **`.reshape(g1.shape)`** devolve os valores de $f$ à forma da malha.
> - **`ax.contour(..., levels=[-1, 0, 1], linestyles=estilos)`** — as curvas em que $f$ vale $-1$, $0$ e $1$ na escala do `SVC`: o hiperplano, contínuo, e as duas bordas da margem, tracejadas.
> - **`modelo.support_vectors_`** — as coordenadas dos vetores de suporte, uma linha por vetor.
> - **`ax.scatter(..., facecolors="none", edgecolors="C3")`** — marcadores vazios, só com o contorno roxo: os anéis.

A figura da margem máxima mostra os três vetores de suporte sobre as bordas da margem, e os outros 37 pontos além delas. O código a seguir empurra cada um desses 37, um de cada vez, 3 unidades para longe do hiperplano, na perpendicular, e refaz o ajuste. Antes de ler a saída, um palpite: o hiperplano muda?

In [ ]:
fora = [i for i in range(40)
    if i not in sv]
iguais = []
for i in fora:
    X_mov = X_sep.copy()
    X_mov[i] += 3 * y[i] * beta
    outro = SVC(kernel="linear",
        C=np.inf).fit(X_mov, y)
    iguais.append(
        np.allclose(outro.coef_,
            modelo.coef_)
        and np.allclose(
            outro.intercept_,
            modelo.intercept_))
print("pontos movidos:",
    len(fora))
print("iguais em todos:",
    all(iguais))

> **🔧 Função**
>
> - **`X_mov[i] += 3 * y[i] * beta`** — anda 3 unidades na direção perpendicular ao hiperplano, para o lado da classe do ponto, isto é, para longe do hiperplano.
> - **`SVC(...).fit(X, y)`** — o `fit` devolve o próprio modelo, e o ajuste cabe na mesma linha da criação. **`np.allclose(a, b)`** — `True` quando todos os elementos dos dois arrays são iguais a menos do arredondamento.

Nos 37 casos, `coef_` e `intercept_` saem iguais aos do ajuste original: empurrados para fora, um de cada vez, esses pontos não alteram o hiperplano. Quem o define são os vetores de suporte, e um ponto que continua além da borda da margem não muda a solução. Dos 40 pontos de treino, só 3 decidem a fronteira.

### Um ponto novo gira a fronteira

Essa dependência de poucos pontos tem um outro lado. Considere um ponto novo da classe $1$ em $(1{,}2;\ 1{,}2)$, entre as duas nuvens. Antes de ler a saída, um palpite pela equação do hiperplano ou pela figura da margem máxima: de que lado do hiperplano ele cai? O código também o põe no treino e refaz o ajuste:

In [ ]:
novo = np.array([[1.2, 1.2]])
f_novo = modelo.decision_function(
    novo)[0]
previsto = modelo.predict(novo)[0]
print("previsto:",
    numero_br(previsto, 0))
print("distância com sinal:",
    numero_br(f_novo / norma))
X_novo = np.vstack([X_sep, novo])
y_novo = np.r_[y, 1]
novo_m = SVC(kernel="linear",
    C=np.inf).fit(X_novo, y_novo)
print("support_:",
    novo_m.support_)
print("n_support_:",
    novo_m.n_support_)
erros = (novo_m.predict(X_novo)
    != y_novo).sum()
print("erros de treino:", erros)
c = novo_m.coef_[0]
M_novo = 1 / np.linalg.norm(c)
print("M novo:",
    numero_br(M_novo))
b_novo = c * M_novo
cos = beta @ b_novo
giro = np.degrees(np.arccos(cos))
print("giro:", numero_br(giro, 1),
    "graus")

> **🔧 Função**
>
> - **`modelo.predict(X)`** — a classe prevista para cada linha de `X`: `classes_[1]` onde $f > 0$, e `classes_[0]` onde $f < 0$.
> - **`np.vstack([X_sep, novo])`** — empilha as linhas: os 40 pontos e o novo embaixo, na posição 40. **`np.r_[y, 1]`** — `y` com o valor 1 no fim.
> - **`beta @ b_novo`** — o produto escalar dos dois vetores de coeficientes, os dois com norma 1: é o cosseno do ângulo entre eles. **`np.arccos`** dá o ângulo em radianos, e **`np.degrees`** o passa para graus.

O ajuste novo, com o antigo por baixo:

In [ ]:
# Figura: Um recorte da figura da margem máxima com o ponto novo, o triângulo laranja maior com contorno cinza, já no treino. A reta cinza contínua é o novo hiperplano de margem máxima, com as bordas da margem tracejadas, e a reta verde pontilhada é o hiperplano antigo, sem o ponto novo. Os anéis roxos marcam os dois vetores de suporte do ajuste novo.
fig, ax = plt.subplots(
    figsize=(3.8, 4.9),
    layout="constrained")
regioes(ax, novo_m)
classes(ax, X_sep, y)
ax.scatter(1.2, 1.2, s=80,
    color="C1", marker="^",
    edgecolors="0.45", zorder=3,
    label="ponto novo")
aneis(ax, novo_m)
ax.axline((0, -beta0 / beta[1]),
    slope=-beta[0] / beta[1],
    color="C2", lw=2, ls=":",
    label="hiperplano antigo")
ax.plot([], [], color="0.45",
    label="hiperplano novo")
ax.plot([], [], color="0.45",
    ls="--", label="margem")
ax.set_xlim(-1, 3.5)
ax.set_ylim(-1, 3.5)
ax.set_xticks(range(-1, 4))
ax.set_yticks(range(-1, 4))
fig.legend(ncols=2,
    loc="outside lower center")
plt.show()

> **🔧 Função**
>
> - **`ax.axline((0, corte), slope=incl)`** — a reta que passa por $(0, \text{corte})$ com inclinação `incl`. Do hiperplano antigo, $\beta_0 + \beta_1 X_1 + \beta_2 X_2 = 0$, sai $X_2 = -\beta_0/\beta_2 - (\beta_1/\beta_2)\,X_1$. **`ls=":"`** desenha a reta pontilhada.
> - **`ax.plot([], [], ..., label=...)`** — uma linha sem pontos, que só serve para pôr na legenda o estilo das retas desenhadas por `contour`.

O hiperplano antigo põe o ponto novo do lado errado, o da classe $-1$, a uma distância de 0,27 da reta, embora ele seja da classe $1$. Com ele no treino, o hiperplano de margem máxima continua sem nenhum erro de treino, mas gira 21,6 graus, e a margem cai de 0,37 para 0,12. O ajuste novo tem dois vetores de suporte, um de cada classe: o ponto 1, da classe $-1$, e o próprio ponto novo, o 40. Um único ponto girou a fronteira de todo o conjunto.

### Quando as classes se misturam

Se as nuvens se misturam, a primeira pergunta deixa de ser a margem: alguma reta ainda separa as classes? Dá para responder sem pressupor a separação, varrendo as direções possíveis da reta, uma a uma, e conferindo se alguma deixa as classes em lados opostos. Antes de usar essa varredura, convém conferi-la num caso em que a resposta é conhecida: o caso separável dos 40 pontos originais, sem o ponto novo, em que o `SVC` já deu a margem.

A margem 0,37 dá para refazer sem o algoritmo, a partir da direção $\beta$ do primeiro ajuste. Projetados em $\beta$, os pontos da classe $1$ ficam a partir de um valor mínimo, e os da classe $-1$ até um valor máximo. A diferença entre o mínimo e o máximo é a **folga**. Se ela é positiva, a reta perpendicular a $\beta$ no ponto médio da folga separa as classes com margem igual à metade dela; se é negativa ou zero, nenhuma reta perpendicular a $\beta$ deixa a classe $1$ do lado para onde $\beta$ aponta.

In [ ]:
def folga(pts, b):
    p = pts @ b
    return (p[y == 1].min()
        - p[y == -1].max())

f_svc = folga(X_sep, beta)
print("folga em β:",
    numero_br(f_svc, 3))
igual = abs(f_svc / 2 - M) < 1e-3
print("metade = M (±0,001):",
    bool(igual))

A folga na direção do `SVC` é 0,745, e a metade dela coincide com a margem $M$ a menos de 0,001. A conta não depende do `SVC`: vale para qualquer direção. As três retas testadas antes eram só três, e todas presas ao centro. A varredura testa 720 direções $\beta = (\cos\theta, \operatorname{sen}\theta)$, de meio em meio grau de 0 a 359,5, cada uma com o seu melhor deslocamento, o ponto médio da folga. As direções opostas, $\theta$ e $\theta + 180^\circ$, cobrem os dois lados em que a classe $1$ pode ficar.

In [ ]:
graus = np.arange(0, 360, 0.5)

def folgas(pts):
    res = []
    for t in np.radians(graus):
        b = [np.cos(t), np.sin(t)]
        res.append(folga(pts, b))
    return np.array(res)

f_sep = folgas(X_sep)
melhor = f_sep.max() / 2
print("direções:", len(graus))
print("maior margem na grade:",
    numero_br(melhor))
print("M do SVC:", numero_br(M))
perto = abs(melhor - M) < 1e-3
print("diferença < 0,001:", perto)
a_grade = graus[f_sep.argmax()]
a_svc = np.degrees(
    np.arctan2(beta[1], beta[0]))
print("ângulo na grade:",
    numero_br(a_grade, 1))
print("ângulo do SVC:",
    numero_br(a_svc, 1))

> **🔧 Função**
>
> - **`np.arange(0, 360, 0.5)`** — os valores de 0 a 359,5, de meio em meio grau.
> - **`f_sep.argmax()`** — a posição do maior valor do array; `graus[...]` dá o ângulo dessa posição.
> - **`np.arctan2(b2, b1)`** — o ângulo, em radianos, do vetor $(b_1, b_2)$ com o eixo horizontal.

A melhor das 720 direções tem margem 0,37, e a diferença para o $M$ do `SVC` fica abaixo de 0,001. O ângulo dela, 25,0°, e o da direção do `SVC`, 24,9°, diferem em 0,1°, menos que o passo da grade: a varredura chega, por tentativa, à mesma margem e à mesma direção que o `SVC` acha resolvendo o problema de maximização.

Com $p = 2$, as direções são os ângulos de um círculo, e varrê-los de meio em meio grau custa 720 contas. Com mais preditores, uma grade igualmente fina precisa de ângulos em $p - 1$ eixos, e o número de direções cresce de forma explosiva a cada preditor novo. Por isso, na prática, o hiperplano de margem máxima sai da solução do problema de maximização, não de uma varredura.

Os mesmos 40 sorteios, `X`, com um deslocamento menor, de 1 em vez de 2,7, deixam as duas nuvens se misturarem. A mesma varredura pelas 720 direções diz se alguma reta ainda as separa:

In [ ]:
X_mis = X.copy()
X_mis[y == 1] += 1
f_mis = folgas(X_mis)
print("alguma direção separa:",
    bool(np.any(f_mis > 0)))
print("maior folga:",
    numero_br(f_mis.max()))
mis = SVC(kernel="linear", C=1000)
mis.fit(X_mis, y)
erros = (mis.predict(X_mis)
    != y).sum()
print("erros de treino:", erros,
    "de", len(y))

> **🔧 Função**
>
> - **`np.any(f_mis > 0)`** — `True` quando ao menos um elemento é positivo.
> - **`SVC(kernel="linear", C=1000)`** — um valor de `C` finito e muito grande; o que `C` faz vem na seção 16.2.

Nenhuma das 720 direções testadas separa as classes, e mesmo a melhor delas tem folga negativa, de −1,51: em cada uma das 720, as projeções das duas classes se sobrepõem. Com `C=np.inf`, o ajuste pediria uma separação que a varredura não encontra; com `C=1000`, ele erra 9 dos 40 pontos de treino. As nuvens misturadas:

In [ ]:
# Figura: Os mesmos 40 sorteios com a classe $1$ deslocada de 1 em vez de 2,7: a classe $-1$ em círculos azuis e a classe $1$ em triângulos laranja. As duas nuvens se sobrepõem.
fig, ax = plt.subplots(
    figsize=(3.8, 4.2),
    layout="constrained")
classes(ax, X_mis, y)
ax.set_xlim(-2.5, 3.5)
ax.set_ylim(-2.5, 3.5)
fig.legend(ncols=2,
    loc="outside lower center")
plt.show()

Com as classes separadas, quem decide onde o hiperplano de margem máxima passa são os vetores de suporte, três dos 40 pontos; os outros podem se afastar da margem sem mudar nada. A mesma dependência deixa a fronteira frágil: um ponto novo perto dela pode girá-la e estreitar a margem. E quando nenhum hiperplano separa as classes, como indicam as 720 direções testadas em `X_mis`, o problema da margem máxima não tem solução com $M > 0$.

## O Classificador de Vetores de Suporte

> **📌 Nota**
>
> Esta seção corresponde às seções 9.2, 9.5 e 9.6.1 de James et al. (2023).

Quando as duas nuvens de pontos se misturam, nenhuma reta deixa todas as observações de uma classe de um lado e todas as da outra do outro. E, mesmo quando uma reta separa, ela pode depender de poucos pontos: um ponto novo perto da fronteira basta para girá-la. Vale aceitar alguns pontos de treino do lado errado em troca de uma fronteira que não dependa tanto de poucos deles? E quantos aceitar?

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("estilo-figuras.mplstyle")

Para imprimir com vírgula decimal:

In [ ]:
numero_br = lambda v, casas=2: (
    f"{v:,.{casas}f}"
    .replace(",", "_")
    .replace(".", ",")
    .replace("_", ".")
    .replace("-", "−")
)

Os pontos são os mesmos 40 sorteios da seção 16.1, com a classe $1$ deslocada de 1 nas duas coordenadas, o que mistura as nuvens. Depois deles, o mesmo gerador sorteia 10.000 pontos de teste, 5.000 de cada classe, com o mesmo deslocamento (o `_` em `10_000` só separa os milhares para quem lê):

In [ ]:
import numpy as np

rng = np.random.default_rng(16)
X = rng.normal(size=(40, 2))
y = np.repeat([-1, 1], 20)
X_mis = X.copy()
X_mis[y == 1] += 1
X_teste = rng.normal(
    size=(10_000, 2))
y_teste = np.repeat(
    [-1, 1], 5_000)
X_teste[y_teste == 1] += 1
print("treino:", X_mis.shape)
print("teste:", X_teste.shape)

### Uma margem que aceita violações

O classificador de margem máxima pede que todo ponto de treino fique do lado certo da margem. Afrouxar essa exigência dá o **classificador de vetores de suporte**: cada ponto ganha uma variável de folga, que mede quanto ele invade a margem, e a soma dessas folgas tem um teto. Essa folga é de cada ponto, e não a folga entre as projeções das duas classes da seção 16.1.

> **🔷 Conceito**
>
> O **classificador de vetores de suporte** escolhe $\beta_0, \dots, \beta_p$, as folgas $\varepsilon_1, \dots, \varepsilon_n$ e a margem $M$ que resolvem
>
> $$
> \begin{aligned}
> &\text{maximizar}\ M \\
> &\text{sujeito a}\ \sum_{j=1}^{p} \beta_j^2 = 1, \\
> &y_i\, f(x_i) \ge M(1 - \varepsilon_i), \\
> &\varepsilon_i \ge 0,\ \ \sum_{i=1}^{n} \varepsilon_i \le C.
> \end{aligned}
> $$
>
> A **folga** $\varepsilon_i$ diz onde o ponto $i$ ficou:
>
> - $\varepsilon_i = 0$: do lado certo da margem, sobre a borda ou além dela;
> - $0 < \varepsilon_i < 1$: dentro da margem, mas do lado certo do hiperplano;
> - $\varepsilon_i = 1$: sobre o hiperplano;
> - $\varepsilon_i > 1$: do lado errado do hiperplano, um erro de treino.
>
> O número $C \ge 0$ é um **orçamento**: o total de folga que as $n$ observações podem gastar. Cada ponto do lado errado do hiperplano gasta mais de 1, então no máximo $C$ pontos ficam do lado errado. Com $C = 0$, nenhuma folga é permitida e o problema volta a ser o da margem máxima.

Na escala em que o `SVC` guarda os coeficientes, a da seção 16.1, a borda da margem é $y_i f(x_i) = 1$ e a restrição com $M$ vira $y_i f(x_i) \ge 1 - \varepsilon_i$. A folga de cada ponto é então $\varepsilon_i = \max(0,\ 1 - y_i f(x_i))$: zero para quem está sobre a borda ou além dela, e $1 - y_i f(x_i)$ para os outros.

> **⚠️ Atenção — O `C` do `SVC` vai no sentido oposto**
>
> O `SVC` tem um argumento chamado `C`, mas ele não é o orçamento $C$: vai no sentido contrário. Um `C` pequeno no `SVC` dá uma margem larga, com muita folga gasta; um `C` grande dá uma margem estreita, com pouca folga. Daqui em diante, `C` em fonte de código é sempre o argumento do `SVC`, e $C$, em itálico, o orçamento.

O `SVC` com `C=1`, o valor padrão do argumento, e o $y_i f(x_i)$ e a folga de cada um dos 40 pontos. Aqui as duas coordenadas estão na mesma escala; com preditores em unidades diferentes, a margem e as folgas dependeriam da unidade, e o `StandardScaler` entra antes do `SVC` num `Pipeline`, ajustado só no treino ([seção 10.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/06-vazamento.html)):

In [ ]:
from sklearn.svm import SVC

modelo = SVC(kernel="linear", C=1)
modelo.fit(X_mis, y)
yf = y * modelo.decision_function(
    X_mis)
eps = np.maximum(0, 1 - yf)
print("maior folga:",
    numero_br(eps.max()))

> **🔧 Função**
>
> - **`np.maximum(0, 1 - yf)`** — compara, elemento a elemento, 0 com $1 - y_i f(x_i)$ e fica com o maior: a folga de cada ponto.

Em termos de $y_i f(x_i)$, as quatro faixas da folga são $y_i f(x_i) \ge 1$, entre 0 e 1, igual a 0 e menor que 0. O `SVC` resolve o problema só até uma tolerância de 0,001, o padrão do argumento `tol`, e por isso o código testa primeiro, com essa folga, quem está sobre a borda ($y_i f(x_i)$ perto de 1) e sobre o hiperplano (perto de 0):

In [ ]:
sobre = np.isclose(yf, 1,
    atol=1e-3)
no_hip = np.isclose(yf, 0,
    atol=1e-3)
nomes = ["sobre a borda",
    "no hiperplano", "além",
    "dentro", "lado errado"]
faixa = np.select(
    [sobre, no_hip, yf > 1,
        yf > 0],
    nomes[:4], nomes[4])
for nome in nomes:
    n = (faixa == nome).sum()
    print(nome + ":", n)
dentro = faixa == "dentro"
errado = faixa == "lado errado"
erros = (modelo.predict(X_mis)
    != y).sum()
print("erros de treino:", erros)
print("erros = lado errado:",
    erros == errado.sum())

> **🔧 Função**
>
> - **`np.select(condicoes, nomes, padrao)`** — para cada ponto, o nome da primeira condição da lista que é `True`, e `padrao` quando nenhuma é: aqui, quem tem $y_i f(x_i)$ negativo. A ordem decide: um ponto sobre a borda tem `yf > 0` também, mas cai em `"sobre a borda"`, que vem antes; e `yf > 0` só pega quem não passou de 1. **`nomes[:4]`** — os quatro primeiros nomes da lista; `nomes[4]`, o quinto, é o padrão.
> - **`faixa == "dentro"`** — `True` onde a faixa do ponto é `"dentro"`.

Na figura, a reta contínua é o hiperplano e as tracejadas são as bordas da margem. Os pontos sem anel estão além da tracejada do seu lado. O anel roxo marca quem está sobre a tracejada ou entre ela e a reta contínua; o verde, quem passou da reta contínua para a região da outra classe:

In [ ]:
# Figura: O classificador de vetores de suporte com `C=1` nos 40 pontos misturados. A reta cinza contínua é o hiperplano, e as tracejadas, as bordas da margem. Anéis roxos: os pontos sobre a borda ($\varepsilon_i = 0$) ou dentro da margem ($0 < \varepsilon_i < 1$). Anéis verdes: os 9 pontos do lado errado do hiperplano, com $\varepsilon_i > 1$, os erros de treino.
import matplotlib.pyplot as plt

def classes(ax, pts, y):
    a = pts[y == -1]
    b = pts[y == 1]
    ax.scatter(a[:, 0], a[:, 1],
        color="C0", s=18,
        label="classe −1")
    ax.scatter(b[:, 0], b[:, 1],
        color="C1", s=24,
        marker="^",
        label="classe 1")
    ax.set_aspect("equal")

def regioes(ax, modelo):
    g = np.linspace(-3, 4, 141)
    g1, g2 = np.meshgrid(g, g)
    malha = np.column_stack(
        [g1.ravel(), g2.ravel()])
    z = modelo.decision_function(
        malha).reshape(g1.shape)
    lim = np.abs(z).max() + 1
    ax.contourf(g1, g2, z,
        levels=[-lim, 0, lim],
        colors=["C0", "C1"],
        alpha=0.2)
    estilos = ["--", "-", "--"]
    ax.contour(g1, g2, z,
        levels=[-1, 0, 1],
        colors="0.45",
        linewidths=1.5,
        linestyles=estilos)

def anel(ax, pts, cor, rotulo,
        tam=90, lw=1.5):
    ax.scatter(pts[:, 0],
        pts[:, 1], s=tam,
        facecolors="none",
        edgecolors=cor,
        linewidths=lw,
        label=rotulo)

fig, ax = plt.subplots(
    figsize=(3.8, 4.8),
    layout="constrained")
regioes(ax, modelo)
classes(ax, X_mis, y)
na_margem = sobre | dentro
anel(ax, X_mis[na_margem], "C3",
    "sobre a borda ou dentro")
anel(ax, X_mis[errado],
    "C2", "lado errado")
ax.set_xlim(-2.5, 3.5)
ax.set_ylim(-2.5, 3.5)
ax.set_xticks([-2, 0, 2])
ax.set_yticks([-2, 0, 2])
ax.set_xlabel("$X_1$")
ax.set_ylabel("$X_2$")
fig.legend(ncols=2,
    loc="outside lower center")
plt.show()

> **🔧 Função**
>
> - **`a | b`** — `True` onde ao menos um dos dois é `True`: `na_margem` marca os pontos sobre a borda ou dentro da margem.

`classes` desenha os pontos das duas classes; `regioes` pinta o lado de cada classe e traça o hiperplano e as bordas da margem; `anel` desenha anéis vazios numa cor e num tamanho escolhidos.

Dos 40 pontos, 18 ficam além da margem e 3 sobre a borda, todos com folga zero; 10 invadem a margem sem cruzar o hiperplano; nenhum fica sobre o hiperplano; e 9 estão do lado errado dele, com $\varepsilon_i > 1$. São exatamente os 9 erros de treino. A faixa entre as tracejadas, vazia na margem máxima, agora tem pontos dentro, e os de anel verde estão na região da outra classe.

Na margem máxima, os vetores de suporte eram os pontos sobre a borda. E agora, quais pontos definem o hiperplano? Um palpite pela figura antes da saída:

In [ ]:
sv = modelo.support_
print("vetores de suporte:",
    len(sv))
print("n_support_:",
    modelo.n_support_)
com_anel = sobre | dentro | errado
mesmos = set(sv) == set(
    np.flatnonzero(com_anel))
print("= pontos com anel:",
    mesmos)

> **🔧 Função**
>
> - **`np.flatnonzero(m)`** — as posições em que a máscara `m` é `True`.

Os 22 vetores de suporte, 11 de cada classe, são exatamente os $3 + 10 + 9$ pontos com anel: os que estão sobre a borda, dentro da margem ou do lado errado do hiperplano. Todo ponto com folga positiva é vetor de suporte.

### O que o `C` do `SVC` faz

O código ajusta o `SVC` com sete valores de `C`, de 0,01 a 10, e mede, em cada um, a margem $M$, o número de vetores de suporte e a soma das folgas, isto é, quanto do orçamento $C$ a solução gastou. Antes de ler a tabela, um palpite: quando o `C` do `SVC` cresce, a margem alarga ou estreita? E os vetores de suporte, aumentam ou diminuem?

In [ ]:
import pandas as pd

def resumo(c):
    m = SVC(kernel="linear", C=c)
    m.fit(X_mis, y)
    f_c = m.decision_function(
        X_mis)
    yf_c = y * f_c
    norma = np.linalg.norm(
        m.coef_[0])
    folgas = np.maximum(
        0, 1 - yf_c)
    return {"c_svc": c,
        "margem": 1 / norma,
        "n_vs": len(m.support_),
        "soma_folgas":
            folgas.sum()}

grade = [0.01, 0.03, 0.1, 0.3,
    1, 3, 10]
tabela = pd.DataFrame(
    [resumo(c) for c in grade])
tabela.style.format(
    {"c_svc": "{:g}",
        "margem": "{:.2f}",
        "soma_folgas": "{:.2f}"},
    decimal=",").hide()

> **🔧 Função**
>
> - **`pd.DataFrame([...])`** — uma tabela com uma linha por dicionário da lista e uma coluna por chave: `c_svc` é o `C` do `SVC`, `margem` é $M$, `n_vs` é o número de vetores de suporte, e `soma_folgas` é $\sum_i \varepsilon_i$.
> - **`tabela.style.format({...}, decimal=",")`** — formata a exibição sem mudar os valores: cada coluna com o seu formato (`"{:.2f}"`, duas casas; `"{:g}"`, sem zeros sobrando) e vírgula decimal. **`.hide()`** — esconde a coluna do índice.

Quando o `C` do `SVC` cresce, a margem estreita, de 3,14 para 0,72, os vetores de suporte caem de 40 para 21 e a soma das folgas cai de 29,87 para 18,46. O código confere que nenhuma das três colunas cresce de uma linha para a seguinte, a menos da tolerância de 0,001 do `SVC`, e compara a última linha com um `C` dez vezes maior, 100:

In [ ]:
for col in ["margem", "n_vs",
        "soma_folgas"]:
    passos = np.diff(tabela[col])
    ok = np.all(passos <= 1e-3)
    print(col, "não cresce:", ok)
u10 = tabela.iloc[6, 1:]
u100 = pd.Series(
    resumo(100)).iloc[1:]
u3 = tabela.iloc[5, 1:]
for nome, u in [("C = 3", u3),
        ("C = 100", u100)]:
    igual = np.allclose(
        u.astype(float),
        u10.astype(float),
        atol=1e-3)
    print(nome, "igual a C = 10:",
        igual)

> **🔧 Função**
>
> - **`np.diff(v)`** — as diferenças entre elementos vizinhos: `v[1] - v[0]`, `v[2] - v[1]`, e assim por diante.
> - **`tabela.iloc[6, 1:]`** — a linha na posição 6 (`C` = 10), da segunda coluna em diante, sem `c_svc`. **`pd.Series(dicionario)`** — o dicionário de `resumo(100)` como uma série, com as chaves como rótulos. **`.astype(float)`** — converte os valores para número de ponto flutuante, para o `np.allclose` comparar.

As três colunas andam juntas, e a soma das folgas mostra o sentido invertido do aviso: o `C` do `SVC` cresce, e o orçamento que a solução gasta encolhe. Com `C` = 3, 10 e 100, a margem, os vetores de suporte e a soma das folgas coincidem, e de 1 para 3 a solução ainda muda: os vetores de suporte passam de 22 para 21. Nestes 40 pontos, nos valores medidos, a partir de `C` = 3 apertar o orçamento não muda mais a solução. Isso é destes dados; em outros, o ponto em que a solução para de mudar é outro.

Quatro valores de `C` da grade, um por painel:

In [ ]:
# Figura: O classificador de vetores de suporte nos 40 pontos misturados com quatro valores do `C` do `SVC`, que cresce da esquerda para a direita e de cima para baixo. A reta cinza contínua é o hiperplano, e as tracejadas, as bordas da margem; os anéis roxos marcam os vetores de suporte. No título de cada painel, o `C` do `SVC` e VS, o número de vetores de suporte. Com `C` maior, a margem estreita e sobram menos vetores de suporte.
fig, eixos = plt.subplots(2, 2,
    figsize=(3.8, 4.2),
    sharex=True, sharey=True,
    layout="constrained")
for c, ax in zip(
        [0.03, 0.1, 0.3, 1],
        eixos.ravel()):
    m = SVC(kernel="linear", C=c)
    m.fit(X_mis, y)
    regioes(ax, m)
    classes(ax, X_mis, y)
    anel(ax, m.support_vectors_,
        "C3", None, tam=40, lw=1)
    ax.set_xlim(-2.5, 3.5)
    ax.set_ylim(-2.5, 3.5)
    ax.set_xticks([-2, 0, 2])
    ax.set_yticks([-2, 0, 2])
    vs = len(m.support_)
    rot = numero_br(c, 2).rstrip(
        "0").rstrip(",")
    ax.set_title(
        f"C = {rot} · VS = {vs}",
        fontsize=9)
fig.supxlabel("$X_1$")
fig.supylabel("$X_2$")
plt.show()

> **🔧 Função**
>
> - **`plt.subplots(2, 2, sharex=True, sharey=True)`** — quatro gráficos em duas linhas e duas colunas, com os mesmos limites de eixo; `eixos.ravel()` os põe numa fila, na ordem de leitura.
> - **`ax.set_title(texto, fontsize=9)`** — o título do painel, em fonte menor. **`fig.supxlabel`** e **`fig.supylabel`** — um rótulo só para os eixos de todos os painéis.

A largura da margem regula o equilíbrio entre viés e variância ([seção 7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-qualidade-do-ajuste-e-vies-variancia.html)). Com a margem larga, de `C` pequeno no `SVC`, muitos pontos a violam: todos eles ajudam a decidir o hiperplano, e um ponto a mais ou a menos pesa pouco. É um classificador de variância baixa, que pode ter viés alto. Com a margem estreita, de `C` grande, menos pontos decidem a fronteira, e o classificador se ajusta mais aos dados de treino.

### Por que o sentido é o oposto

O sentido invertido sai de uma outra forma do mesmo problema, com o formato que a regressão ridge ([seção 11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-regressao-ridge.html)) já tinha: uma **perda**, que mede quanto o modelo erra no treino, mais uma **penalidade**, que encolhe os coeficientes. Nela, o classificador de vetores de suporte escolhe $\beta_0, \dots, \beta_p$ sem a restrição $\sum_j \beta_j^2 = 1$ e sem $M$:

$$
\begin{aligned}
&\underset{\beta_0, \dots, \beta_p}{\text{minimizar}} \\
&\sum_{i=1}^{n} \max[0,\ 1 - y_i f(x_i)] \\
&+ \lambda \sum_{j=1}^{p} \beta_j^2.
\end{aligned}
$$

A perda é a soma das folgas, e a penalidade é a da ridge. Com $\lambda$ grande, encolher os coeficientes vale mais que reduzir as folgas: os coeficientes saem pequenos, a margem $1/\lVert\beta\rVert$ sai larga, e a soma das folgas cresce. O `SVC` minimiza

$$
\begin{aligned}
&\tfrac{1}{2} \sum_{j=1}^{p} \beta_j^2 \\
&+ \texttt{C} \sum_{i=1}^{n} \max[0,\ 1 - y_i f(x_i)].
\end{aligned}
$$

Dividir tudo por `C` não muda onde fica o mínimo, e dá a forma anterior com $\lambda = 1/(2\,\texttt{C})$: o `C` do `SVC` está no denominador. Com `C=0.1`, por exemplo, o $\lambda$ correspondente é $1/(2 \cdot 0{,}1) = 5$.

Dá para conferir a conta. Se ela está certa, nenhuma combinação de coeficientes perto da solução do `SVC` dá um valor menor na forma com $\lambda = 5$. O código mexe um pouco nos três coeficientes, 1.000 vezes ao acaso, e conta quantas dessas variações dão um valor menor. Para contraste, faz o mesmo com $\lambda = 10$, o valor que sairia esquecendo o $\tfrac12$. O `scikit-learn` não calcula o valor da forma com $\lambda$, e a função `objetivo` existe só para essa conferência. `m.intercept_[0]` e `m.coef_[0]` estão na escala do `SVC`, sem dividir pela norma, que é a escala em que as duas formas acima estão escritas:

In [ ]:
m = SVC(kernel="linear", C=0.1)
m.fit(X_mis, y)
b0 = m.intercept_[0]
b = m.coef_[0]

def objetivo(b0, b, lam):
    f = b0 + X_mis @ b
    folga = np.maximum(
        0, 1 - y * f)
    return (folga.sum()
        + lam * (b**2).sum())

viz = np.random.default_rng(160)
d = viz.normal(scale=0.05,
    size=(1000, 3))
for lam in [5, 10]:
    base = objetivo(b0, b, lam)
    menores = sum(
        objetivo(b0 + e[0],
            b + e[1:], lam) < base
        for e in d)
    print(f"λ = {lam}: vizinhos",
        "menores:", menores)

> **🔧 Função**
>
> - **`viz.normal(scale=0.05, size=(1000, 3))`** — 1.000 linhas de 3 sorteios da normal com média 0 e desvio padrão 0,05, de um gerador com semente própria, 160, diferente da dos dados.

Com $\lambda = 5$, nenhum dos 1.000 vizinhos fica abaixo da solução do `SVC`; com $\lambda = 10$, 345 ficam, e a mesma solução deixa de ser o mínimo. A correspondência completa, lida nos dois sentidos:

> **📌 Nota**
>
> Um `C` grande no `SVC` é um $\lambda = 1/(2\,\texttt{C})$ pequeno, que é um orçamento $C$ pequeno.
>
> | | orçamento $C$ | `C` do `SVC` |
> |---|---|---|
> | margem larga | grande | pequeno |
> | margem estreita | pequeno | grande |
> | nenhuma violação | $C = 0$ | `C=np.inf` |

A última linha é a da seção 16.1: o orçamento zero corresponde a `C=np.inf`, e só tem solução quando as classes se separam. O zero no argumento do `SVC` não é aceito:

In [ ]:
try:
    SVC(kernel="linear",
        C=0).fit(X_mis, y)
except ValueError as erro:
    msg = str(erro)
    for k in range(0, 90, 30):
        print(msg[k:k + 30])

A mensagem diz que `C` precisa estar no intervalo $(0, \infty]$: `C=np.inf` é aceito, `C=0` não.

### Só os vetores de suporte contam

Na seção 16.1, empurrar para longe do hiperplano um ponto que não era vetor de suporte não mudava a solução. Vale o mesmo com folgas? E se o ponto empurrado for um vetor de suporte? O código pega o ponto mais afastado da margem entre os 18 que não são vetores de suporte e o vetor de suporte de maior folga, o que está mais fundo do lado errado. Cada um é empurrado 2 unidades na direção da sua classe, perpendicular ao hiperplano, e o ajuste é refeito. Antes de ler a saída, um palpite: qual dos dois muda o hiperplano?

In [ ]:
norma = np.linalg.norm(
    modelo.coef_[0])
direcao = modelo.coef_[0] / norma
fora = np.setdiff1d(
    np.arange(40), sv)
i = fora[np.argmax(yf[fora])]
j = sv[np.argmax(eps[sv])]
print("não é VS:", i, "· VS:", j)
print(f"folga de {j}:",
    numero_br(eps[j]))

def empurra(k):
    X_mov = X_mis.copy()
    X_mov[k] += 2 * y[k] * direcao
    m = SVC(kernel="linear", C=1)
    return m.fit(X_mov, y), X_mov

m_i, _ = empurra(i)
iguais = (
    np.allclose(m_i.coef_,
        modelo.coef_, atol=1e-3)
    and np.allclose(
        m_i.intercept_,
        modelo.intercept_,
        atol=1e-3))
print(f"igual, movendo {i}:",
    iguais)
antes = modelo.coef_[0]
print("coef_ antes:",
    numero_br(antes[0]), ";",
    numero_br(antes[1]))
m_j, X_j = empurra(j)
depois = m_j.coef_[0]
print("coef_ depois:",
    numero_br(depois[0]), ";",
    numero_br(depois[1]))
f_j = m_j.decision_function(
    X_j[[j]])[0]
yf_j = y[j] * f_j
print("y f de", j, "depois:",
    numero_br(yf_j))
print(j, "ainda é VS:",
    j in m_j.support_)

> **🔧 Função**
>
> - **`np.setdiff1d(a, b)`** — os elementos de `a` que não estão em `b`: as posições dos pontos que não são vetores de suporte.
> - **`fora[np.argmax(yf[fora])]`** — entre esses, o de maior $y_i f(x_i)$, o mais afastado da margem; **`sv[np.argmax(eps[sv])]`**, entre os vetores de suporte, o de maior folga.
> - **`return m.fit(X_mov, y), X_mov`** — devolve dois valores, o modelo ajustado e os pontos movidos; `m_i, _ = ...` guarda o primeiro e descarta o segundo no `_`. **`X_j[[j]]`** — a linha `j` como uma tabela de uma linha, o formato que o `decision_function` espera.

O ponto 17, que não é vetor de suporte, sai do lugar sem mudar os coeficientes nem o intercepto. O ponto 26, um vetor de suporte com folga 2,30, empurrado as mesmas 2 unidades na direção da sua classe, atravessa o hiperplano e sai da margem: no ajuste novo, o seu $y_i f(x_i)$ é 1,35, e ele deixa de ser vetor de suporte. Os coeficientes mudam de $(1{,}12;\ 0{,}67)$ para $(1{,}23;\ 0{,}74)$.

A análise discriminante linear da [seção 9.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/07-leitura-complementar-lda-e-qda.html) se comporta de outro jeito: a fronteira dela depende das médias das classes e da covariância, calculadas com todos os pontos de treino, inclusive os que estão muito longe dela. O classificador de vetores de suporte ignora os pontos que estão folgadamente do lado certo da margem.

### Escolhendo `C` por validação cruzada

O `C` é um parâmetro de ajuste como o $\lambda$ da ridge, e a escolha segue o mesmo caminho: validação cruzada numa grade de valores ([seção 10.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/05-validacao-cruzada-em-classificacao.html)). A grade é a mesma de antes, e o critério é a taxa de erro média nos 5 grupos:

In [ ]:
from sklearn.model_selection \
    import GridSearchCV
from sklearn.model_selection \
    import StratifiedKFold

grupos = StratifiedKFold(5,
    shuffle=True, random_state=16)
busca = GridSearchCV(
    SVC(kernel="linear"),
    {"C": grade}, cv=grupos)
busca.fit(X_mis, y)
r = busca.cv_results_
erro_cv = 1 - r["mean_test_score"]
def rot(c):
    return f"{c:g}".replace(
        ".", ",")

print("pontos por grupo:",
    len(X_mis) // 5)
print("C: erro de val. cruzada")
for c, e in zip(grade, erro_cv):
    print(rot(c) + ":",
        numero_br(e, 3))
c_best = busca.best_params_["C"]
print("escolhido:", rot(c_best))
previsto = busca.predict(X_teste)
teste = (previsto
    != y_teste).mean()
print("erro de teste:",
    numero_br(teste, 4))

> **🔧 Função**
>
> - **`StratifiedKFold(n_splits, shuffle, random_state)`** — divide as linhas em `n_splits` grupos com a mesma proporção de classes do conjunto todo; `shuffle=True` embaralha antes, com a semente `random_state`.
> - **`GridSearchCV(estimador, grade, cv)`**, de `sklearn.model_selection` — ajusta o estimador com cada valor da grade, um dicionário com o nome do argumento e a lista de valores, e mede a taxa de acerto por validação cruzada nos grupos de `cv`. No fim, reajusta com todas as linhas o valor de maior acerto médio; `busca.predict` usa esse ajuste.
> - **`busca.cv_results_`** — um dicionário com os resultados; `r["mean_test_score"]` é o acerto médio nos grupos, na ordem da grade, e `1 -` o transforma em taxa de erro. **`busca.best_params_`** — o valor escolhido, num dicionário.

A validação cruzada escolhe `C` = 0,01, a margem mais larga da grade, com erro de validação cruzada 0,175, e a taxa de erro de teste desse ajuste é 0,2317. O valor está na ponta da grade, e isso levanta uma dúvida: um `C` ainda menor faria melhor? O código olha o ajuste com `C` = 0,01 e o compara com `C` = 0,001:

In [ ]:
from sklearn.model_selection \
    import cross_val_score

m01 = SVC(kernel="linear", C=0.01)
m01.fit(X_mis, y)
print("vetores de suporte:",
    len(m01.support_))
pesos = np.abs(m01.dual_coef_)
print("todos com peso C:",
    np.allclose(pesos, 0.01))
m001 = SVC(kernel="linear",
    C=0.001).fit(X_mis, y)
def reta(m):
    v = np.r_[m.intercept_,
        m.coef_[0]]
    return v / np.linalg.norm(
        m.coef_[0])
print("C = 0,001, mesma reta:",
    np.allclose(reta(m01),
        reta(m001)))
acerto = cross_val_score(
    SVC(kernel="linear", C=0.001),
    X_mis, y, cv=grupos)
e001 = 1 - acerto.mean()
print("val. cruzada, C = 0,001:",
    numero_br(e001, 3))

> **🔧 Função**
>
> - **`m.dual_coef_`** — o peso de cada vetor de suporte na solução, com o sinal da classe. Em valor absoluto, fica entre 0 e o `C` do `SVC`.
> - **`cross_val_score(estimador, X, y, cv)`**, de `sklearn.model_selection` — a taxa de acerto em cada um dos grupos de `cv`; com os mesmos `grupos` da busca, o erro médio é comparável ao da tabela acima.
> - **`reta(m)`** — junta o intercepto e os coeficientes num vetor só e o divide pela norma de $(\beta_1, \beta_2)$, a escala de $\sum_j \beta_j^2 = 1$ da seção 16.1: a mesma reta dá o mesmo vetor, qualquer que seja a escala do ajuste.

Com `C` = 0,01, os 40 pontos são vetores de suporte, e todos com o mesmo peso, o maior permitido. O orçamento é tão folgado que todo ponto entra na decisão por igual, e diminuir o `C` para 0,001 dá a mesma reta. Com os mesmos grupos, o erro de validação cruzada de `C` = 0,001 é 0,175, igual ao de 0,01: estender a grade para baixo criaria um empate, e o `GridSearchCV`, que fica com o primeiro valor da grade entre os empatados, trocaria só o valor de `C`, não o classificador escolhido.

Estes dados foram simulados, e por isso a regra de menor taxa de erro esperada é conhecida: o **classificador de Bayes** da [seção 7.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/07-classificacao-e-o-classificador-de-bayes.html). As duas classes são normais com o mesmo espalhamento, têm o mesmo tamanho, e a classe $1$ foi deslocada de 1 nas duas coordenadas; com isso, a fronteira de Bayes é a reta $X_1 + X_2 = 1$. Nos mesmos pontos de teste, ela erra:

In [ ]:
lado = X_teste.sum(axis=1) > 1
regra = np.where(lado, 1, -1)
bayes = (regra != y_teste).mean()
print("regra X1 + X2 > 1:",
    numero_br(bayes, 4))

> **🔧 Função**
>
> - **`np.where(cond, 1, -1)`** — 1 onde `cond` é `True`, e −1 onde é `False`.

Nestes pontos, a regra de Bayes erra 0,2304, e o `SVC` escolhido, 0,2317: a diferença, de 0,0013, é pequena. Mas a escolha de `C` não é firme. Com outra semente no embaralhamento dos grupos, a validação cruzada pode escolher outro valor. O código repete a busca com as sementes de 0 a 19 e mede o erro de teste de cada `C` da grade:

In [ ]:
escolhas = []
for s in range(20):
    g = StratifiedKFold(5,
        shuffle=True,
        random_state=s)
    b = GridSearchCV(
        SVC(kernel="linear"),
        {"C": grade}, cv=g)
    b.fit(X_mis, y)
    escolhas.append(
        b.best_params_["C"])
conta = pd.Series(escolhas)
print("C: vezes escolhido")
for c, n in (conta.value_counts()
        .sort_index().items()):
    print(rot(c) + ":", n)
erros_t = []
for c in grade:
    m = SVC(kernel="linear", C=c)
    m.fit(X_mis, y)
    p = m.predict(X_teste)
    erros_t.append(
        (p != y_teste).mean())
erros_t = np.array(erros_t)
entre = np.all((erros_t > 0.23)
    & (erros_t < 0.26))
print("teste entre 0,23 e 0,26:",
    entre)

> **🔧 Função**
>
> - **`pd.Series(escolhas).value_counts()`** — quantas vezes cada valor aparece; **`.sort_index()`** ordena pelos valores de `C`, e **`.items()`** percorre os pares (valor, contagem).
> - **`a & b`** — `True` onde os dois são `True`: `entre` confere que todo erro de teste está acima de 0,23 e abaixo de 0,26.

As 20 sementes escolhem 6 valores diferentes de `C`, de 0,01 a 10. Nestes dados, nenhum `C` da grade é muito pior que outro: o erro de teste de todos fica entre 0,23 e 0,26. A curva de validação cruzada, ao contrário, é ruidosa: com 8 pontos por grupo, um erro a mais num grupo soma $1/8$ ao erro dele e $1/40 = 0{,}025$ à média dos 5. Com 40 pontos de treino, pequenas mudanças nos grupos trocam o vencedor.

### A perda dobradiça e a regressão logística

Na forma com $\lambda$, a perda do classificador de vetores de suporte é a soma de $\max[0,\ 1 - y_i f(x_i)]$, a **perda dobradiça** (*hinge loss*): uma linha que desce em reta até $y_i f(x_i) = 1$ e daí em diante é zero, com uma dobra no 1. Na ridge, a perda é a soma dos quadrados dos resíduos.

A regressão logística da [seção 9.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-regressao-logistica.html) também pode ser escrita com uma perda. Com as classes em $-1$ e $1$, a perda de cada ponto é $\log(1 + e^{-y_i f(x_i)})$. As duas perdas, calculadas em quatro valores de $y_i f(x_i)$:

In [ ]:
z = np.array([-2, 0, 1, 3])
dobra = np.maximum(0, 1 - z)
logis = np.log1p(np.exp(-z))
print("yf: dobradiça · logística")
for a, b1, b2 in zip(z, dobra,
        logis):
    print(numero_br(a, 0), ":",
        numero_br(b1, 0), "·",
        numero_br(b2, 4))

> **🔧 Função**
>
> - **`np.exp(-z)`** — $e^{-z}$, elemento a elemento. **`np.log1p(v)`** — $\log(1 + v)$, o logaritmo natural.

Em $y_i f(x_i) = 1$ e em 3, a perda dobradiça é zero, e a logística, 0,3133 e 0,0486: pequena, mas não zero. O gráfico das duas:

In [ ]:
# Figura: A perda dobradiça, em laranja, e a perda da regressão logística, em azul, como funções de $y_i f(x_i)$. A linha vertical cinza marca $y_i f(x_i) = 1$, a borda da margem: dali para a direita, a perda dobradiça é zero, e a logística continua positiva, cada vez menor.
zz = np.linspace(-4, 3, 281)
fig, ax = plt.subplots(
    figsize=(3.8, 2.8),
    layout="constrained")
ax.plot(zz,
    np.maximum(0, 1 - zz),
    color="C1", lw=2)
ax.plot(zz,
    np.log1p(np.exp(-zz)),
    color="C0", lw=2)
ax.axvline(1, color="0.45",
    ls="--", lw=1)
ax.text(-1.6, 3.2, "dobradiça",
    color="C1")
ax.text(-3.4, 1.7, "logística",
    color="C0")
ax.set_xlabel("$y_i f(x_i)$")
ax.set_ylabel("perda")
ax.set_xlim(-4, 3)
ax.set_ylim(-0.1, 5)
plt.show()

> **🔧 Função**
>
> - **`ax.plot(x, y, color, lw)`** — uma linha pelos pontos, com a cor e a espessura `lw`. **`ax.axvline(1, ...)`** — uma linha vertical em $x = 1$.
> - **`ax.text(x, y, texto, color)`** — escreve o texto na posição $(x, y)$, nas coordenadas dos dados.

As duas curvas têm a mesma forma geral: grandes para pontos do lado errado, perto de zero para os bem classificados. A diferença está no zero exato. Com a perda dobradiça, um ponto além da margem não contribui em nada, e é por isso que só os vetores de suporte decidem o hiperplano. Com a logística, todo ponto contribui um pouco. A mesma ideia de ignorar os pontos bem resolvidos vale para respostas quantitativas: a regressão por vetores de suporte, o `SVR` do `scikit-learn`, usa uma perda que é zero quando o resíduo, em valor absoluto, fica abaixo de uma tolerância, e cresce só com os resíduos maiores que ela.

Com perdas parecidas, os dois métodos costumam dar fronteiras parecidas, mesmo com a penalidade do `SVC` com `C=1`, $\lambda = 1/(2 \cdot 1) = 0{,}5$, e nenhuma na logística. O código ajusta a regressão logística sem penalidade aos mesmos 40 pontos e compara as previsões de teste com as do `SVC` com `C=1`:

In [ ]:
from sklearn.linear_model import (
    LogisticRegression)

logit = LogisticRegression(
    C=np.inf,
    solver="newton-cholesky",
    tol=1e-8)
logit.fit(X_mis, y)
p_log = logit.predict(X_teste)
p_svc = modelo.predict(X_teste)
e_log = (p_log != y_teste).mean()
e_svc = (p_svc != y_teste).mean()
print("teste, logística:",
    numero_br(e_log, 4))
print("teste, SVC:",
    numero_br(e_svc, 4))
conc = (p_log == p_svc).mean()
print("mesma classe:",
    numero_br(conc, 4))
print("β0; β1; β2 (Σβj² = 1):")
for nome, m in [
        ("logística", logit),
        ("SVC", modelo)]:
    v = reta(m)
    print(nome,
        numero_br(v[0], 3), ";",
        numero_br(v[1], 3), ";",
        numero_br(v[2], 3))

> **🔧 Função**
>
> - **`LogisticRegression(C=np.inf, solver="newton-cholesky", tol=1e-8)`**, de `sklearn.linear_model` — a regressão logística sem penalidade (`C=np.inf`), resolvida pelo método de Newton com tolerância pequena (`tol=1e-8`), que chega ao mesmo ponto em máquinas diferentes.

As duas fronteiras dão a mesma classe a 0,9796 dos pontos de teste, e os erros de teste, 0,2403 na logística e 0,2391 no `SVC`, diferem em 0,0012. Na escala de $\sum_j \beta_j^2 = 1$, os coeficientes das duas retas ficam próximos, sem coincidir.

### O `LinearSVC`

O `scikit-learn` tem um segundo classificador por hiperplano, o `LinearSVC`. O nome sugere o mesmo `SVC` com `kernel="linear"`, mas o padrão dele resolve outro problema: a perda é o quadrado da dobradiça, e o intercepto entra na penalidade, junto com os coeficientes. Com `loss="hinge"`, a perda passa a ser a dobradiça, mas o intercepto continua penalizado:

In [ ]:
from sklearn.svm import LinearSVC

def mostra(nome, m):
    b0 = m.intercept_[0]
    b = m.coef_[0]
    print(nome)
    print("  β0:", numero_br(b0),
        "· β:", numero_br(b[0]),
        ";", numero_br(b[1]))

padrao = LinearSVC(C=1)
padrao.fit(X_mis, y)
dobradica = LinearSVC(C=1,
    loss="hinge", random_state=16)
dobradica.fit(X_mis, y)
mostra("SVC linear", modelo)
for nome, m in [
        ("LinearSVC", padrao),
        ("hinge", dobradica)]:
    mostra(nome, m)
    p = m.predict(X_teste)
    igual = (p == p_svc).mean()
    print("  mesma classe:",
        numero_br(igual, 4))

> **🔧 Função**
>
> - **`LinearSVC(C, loss, random_state)`**, de `sklearn.svm` — um classificador por hiperplano com outro algoritmo de ajuste. O padrão, `loss="squared_hinge"`, usa o quadrado da perda dobradiça; `loss="hinge"`, a própria dobradiça. O intercepto é penalizado nos dois casos. Com `loss="hinge"`, o algoritmo percorre as observações numa ordem sorteada, e `random_state` fixa esse sorteio.

Os coeficientes saem outros: $(0{,}57;\ 0{,}39)$ no padrão e $(1{,}08;\ 0{,}65)$ com a dobradiça, contra $(1{,}12;\ 0{,}67)$ do `SVC`. As previsões de teste concordam com as do `SVC` em 0,9828 e 0,9892 dos pontos. Para ler a margem, as folgas e os vetores de suporte como nesta seção, o estimador é o `SVC` com `kernel="linear"`.

### Quantos pontos aceitar

Vale aceitar pontos de treino do lado errado, e quantos? Não há um número fixo: o orçamento é um parâmetro de ajuste, escolhido por validação cruzada como o $\lambda$ da ridge. Nestes 40 pontos, a validação cruzada escolheu a margem mais larga da grade, em que todos os pontos são vetores de suporte, mas sem firmeza: em 10 das 20 divisões dos grupos, o escolhido foi `C` = 0,01, e nas outras 10, outro valor; e o erro de teste de todo `C` da grade ficou entre 0,23 e 0,26. O orçamento limita quantos pontos ficam do lado errado, no máximo $C$, mas não fixa esse número: é o ajuste que decide quantos cruzam o hiperplano. Em troca dos pontos do lado errado, a fronteira passa a depender de muitos pontos, e não de poucos.

## Kernels

> **📌 Nota**
>
> Esta seção corresponde às seções 9.3, 9.6.2 e 9.6.3 de James et al. (2023).

Uma classe se concentra no meio do plano; a outra fica em duas nuvens, uma de cada lado. Uma reta só divide o plano em dois lados, e a classe do meio está entre as duas nuvens da outra. Dá para manter a margem, as folgas e os vetores de suporte e, mesmo assim, ganhar uma fronteira curva?

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("estilo-figuras.mplstyle")

### Quando a fronteira não é reta

Os pontos são sorteados de normais com desvio padrão 1 nas duas coordenadas. A primeira metade das linhas é deslocada para perto de $(2; 2)$, e o quarto seguinte, para perto de $(-2; -2)$: esses três quartos formam a classe $-1$. O último quarto fica em torno da origem e forma a classe $1$. São 200 pontos de treino e, sorteados depois deles, 10.000 de teste. Em `X[: n // 2] += 2`, o `: n // 2` pega as linhas da primeira até a de posição `n // 2 - 1`, e as duas colunas delas recebem 2 a mais:

In [ ]:
import numpy as np

rng = np.random.default_rng(16)

def nuvens(n):
    X = rng.normal(size=(n, 2))
    X[: n // 2] += 2
    X[n // 2 : 3 * n // 4] -= 2
    i = np.arange(n)
    y = np.where(i < 3 * n // 4,
        -1, 1)
    return X, y

X, y = nuvens(200)
X_teste, y_teste = nuvens(10_000)
print("classe −1:",
    (y == -1).sum())
print("classe 1:", (y == 1).sum())

Para imprimir os resultados com vírgula decimal:

In [ ]:
numero_br = lambda v, casas=2: (
    f"{v:,.{casas}f}"
    .replace(",", "_")
    .replace(".", ",")
    .replace("_", ".")
    .replace("-", "−")
)

O primeiro candidato é o classificador de vetores de suporte da seção 16.2, com `C=1`:

In [ ]:
from sklearn.svm import SVC

linear = SVC(kernel="linear", C=1)
linear.fit(X, y)
prev = linear.predict(X_teste)
classes_prev, vezes = np.unique(
    prev, return_counts=True)
print("classes previstas:",
    classes_prev)
print("vezes:", vezes)
erro = (prev != y_teste).mean()
print("erro de teste:",
    numero_br(erro, 4))
maior = np.abs(linear.coef_).max()
print("|coef_| abaixo de 0,001:",
    maior < 1e-3)
print("intercept_:", numero_br(
    linear.intercept_[0]))

> **🔧 Função**
>
> - **`np.unique(v, return_counts=True)`** — os valores distintos de `v`, em ordem, e quantas vezes cada um aparece.

Os 10.000 pontos de teste recebem a classe $-1$, e o erro de teste, 0,25, é a fração da classe $1$, um quarto dos pontos por construção. Os coeficientes saem quase zero e o intercepto, $-1{,}00$: $f(x)$ fica perto de $-1$ no plano inteiro. Os dados e as regiões desse classificador:

In [ ]:
# Figura: Em cima, os 200 pontos de treino: os círculos azuis, da classe $-1$, em duas nuvens, e os triângulos laranja, da classe $1$, entre elas. Embaixo, as regiões do classificador de vetores de suporte com `C=1`: o plano inteiro fica com a cor da classe $-1$.
import matplotlib.pyplot as plt

g = np.linspace(-5, 5, 201)
g1, g2 = np.meshgrid(g, g)
malha = np.column_stack(
    [g1.ravel(), g2.ravel()])

def classes(ax, pts, y):
    a = pts[y == -1]
    b = pts[y == 1]
    ax.scatter(a[:, 0], a[:, 1],
        color="C0", s=12,
        label="classe −1")
    ax.scatter(b[:, 0], b[:, 1],
        color="C1", s=16,
        marker="^",
        label="classe 1")
    ax.set_aspect("equal")
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_xticks([-4, 0, 4])
    ax.set_yticks([-4, 0, 4])
    ax.set_ylabel("$X_2$")

def regioes(ax, modelo,
        margens=True):
    z = modelo.decision_function(
        malha).reshape(g1.shape)
    lim = np.abs(z).max() + 1
    ax.contourf(g1, g2, z,
        levels=[-lim, 0, lim],
        colors=["C0", "C1"],
        alpha=0.2)
    if margens:
        ax.contour(g1, g2, z,
            levels=[-1, 0, 1],
            colors="0.45",
            linewidths=1.5,
            linestyles=["--", "-",
                "--"])

fig, eixos = plt.subplots(2, 1,
    figsize=(3.8, 6.6),
    sharex=True,
    layout="constrained")
classes(eixos[0], X, y)
regioes(eixos[1], linear,
    margens=False)
classes(eixos[1], X, y)
eixos[0].set_title("os dados",
    fontsize=9)
eixos[1].set_title(
    "SVC linear, C = 1",
    fontsize=9)
eixos[1].set_xlabel("$X_1$")
handles, rotulos = (eixos[0]
    .get_legend_handles_labels())
fig.legend(handles, rotulos,
    ncols=2,
    loc="outside lower center")
plt.show()

> **🔧 Função**
>
> - **`ax.get_legend_handles_labels()`** — os marcadores e os rótulos que um gráfico poria na legenda; passados a `fig.legend`, aparecem uma vez só para a figura toda.

No painel de baixo, a cor do fundo é a classe que o classificador daria a cada ponto do plano. Não há fronteira a traçar: $f$ não muda de sinal. Uma reta que separe a nuvem do meio de uma das nuvens de fora deixa a outra nuvem de fora do mesmo lado que a do meio, e o ajuste prefere não separar nada.

Nas figuras seguintes, a leitura é a mesma, e as curvas cinza marcam a fronteira $f(x) = 0$, contínua, e as bordas da margem, $f(x) = \pm 1$, tracejadas.

Na regressão, uma relação curva entre preditor e resposta se ajusta com termos como $X^2$ e $X^3$ ([seção 8.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/05-interacao-e-termos-nao-lineares.html)). O classificador de vetores de suporte aceita a mesma ideia: em vez de $X_1$ e $X_2$, ele recebe também $X_1^2$, $X_1 X_2$ e $X_2^2$ e procura um hiperplano nesse espaço de 5 colunas. Nele, a fronteira é plana; no plano original, $f(x) = 0$ vira uma equação do segundo grau em $X_1$ e $X_2$, e a curva que ela descreve pode separar a nuvem do meio das outras duas.

In [ ]:
from sklearn.pipeline import (
    make_pipeline)
from sklearn.preprocessing \
    import PolynomialFeatures

ampliado = make_pipeline(
    PolynomialFeatures(2,
        include_bias=False),
    SVC(kernel="linear", C=1))
ampliado.fit(X, y)
print(ampliado[0]
    .get_feature_names_out(
        ["X1", "X2"]))
p_amp = ampliado.predict(X_teste)
e_amp = (p_amp != y_teste).mean()
print("erro de teste:",
    numero_br(e_amp, 4))

> **🔧 Função**
>
> - **`PolynomialFeatures(2, include_bias=False)`**, de `sklearn.preprocessing` — acrescenta às colunas os termos até o grau 2, o primeiro argumento: os quadrados e os produtos dois a dois. `include_bias=False` não cria a coluna de uns, porque o intercepto já é do `SVC`.
> - **`make_pipeline(passo1, passo2)`**, de `sklearn.pipeline` — encadeia os passos: o `fit` e o `predict` passam os dados pelo primeiro e entregam o resultado ao segundo. **`ampliado[0]`** é o primeiro passo.
> - **`.get_feature_names_out(["X1", "X2"])`** — os nomes das colunas que o passo produz, a partir dos nomes dados às colunas de entrada.

Com as 5 colunas, o erro de teste cai de 0,25 para 0,1035. A fronteira, desenhada no plano original:

In [ ]:
# Figura: O classificador de vetores de suporte com `C=1` nas 5 colunas de grau até 2, desenhado no plano de $X_1$ e $X_2$. A curva cinza contínua é a fronteira $f(x) = 0$, e as tracejadas, as bordas da margem, $f(x) = \pm 1$.
fig, ax = plt.subplots(
    figsize=(3.8, 3.9),
    layout="constrained")
regioes(ax, ampliado)
classes(ax, X, y)
ax.set_xlabel("$X_1$")
fig.legend(handles, rotulos,
    ncols=2,
    loc="outside lower center")
plt.show()

No espaço de 5 colunas, a fronteira é um hiperplano; no plano original, são duas curvas, e a faixa laranja entre elas contém a nuvem do meio e deixa as duas nuvens azuis de fora, uma de cada lado. Só que o número de colunas cresce depressa: com $p$ preditores, o grau 2 já dá $p$ quadrados e $p(p-1)/2$ produtos, e os graus maiores crescem mais depressa.

### Produtos internos e kernels

Como chegar a uma fronteira como essa sem montar as colunas? A resposta passa pelo **produto interno** de duas observações, o produto escalar da [seção 4.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/01-vetores.html): a soma dos produtos coordenada a coordenada. Para os pontos $a = (1; 2)$ e $b = (3; -1)$:

In [ ]:
a = np.array([1, 2])
b = np.array([3, -1])
print("<a, b> =", a @ b)

A conta é $1 \cdot 3 + 2 \cdot (-1) = 1$. Com $p$ coordenadas,

$$
\langle x_i, x_{i'} \rangle = \sum_{j=1}^{p} x_{ij}\, x_{i'j}.
$$

A solução do classificador de vetores de suporte pode ser escrita só com produtos internos. Com $S$ o conjunto dos vetores de suporte,

$$
f(x) = \beta_0 + \sum_{i \in S} \alpha_i \langle x, x_i \rangle,
$$

com um número $\alpha_i$ para cada vetor de suporte; para os outros pontos de treino, $\alpha_i = 0$. No `SVC`, os $\alpha_i$ estão em `dual_coef_`, com o sinal da classe de cada vetor de suporte já embutido, e $\beta_0$ está em `intercept_`.

No ajuste e na previsão, os pontos só aparecem dentro de produtos internos: no ajuste, entre os pares de pontos de treino; na previsão, entre o ponto novo $x$ e cada vetor de suporte. A ideia é trocar o produto interno por outra função de duas observações, $K(x_i, x_{i'})$, chamada **kernel** (núcleo), que mede quanto as duas se parecem. O produto interno é o **kernel linear**, e com ele se volta ao classificador de vetores de suporte.

Com outro kernel na mesma soma, sai um classificador novo:

> **🔷 Conceito**
>
> Com um kernel $K$, a **máquina de vetores de suporte** (SVM, de *support vector machine*) usa
>
> $$
> f(x) = \beta_0 + \sum_{i \in S} \alpha_i\, K(x, x_i).
> $$
>
> O **kernel polinomial** de grau $d$ é
>
> $$
> \begin{aligned}
> &K(x_i, x_{i'}) = \\
> &\Big(1 + \sum_{j=1}^{p} x_{ij}\, x_{i'j}\Big)^{d},
> \end{aligned}
> $$
>
> e o **kernel radial**, com $\gamma > 0$, é
>
> $$
> \begin{aligned}
> &K(x_i, x_{i'}) = \\
> &\exp\Big(-\gamma \sum_{j=1}^{p} (x_{ij} - x_{i'j})^2\Big).
> \end{aligned}
> $$

No radial, a soma do expoente é o quadrado da distância euclidiana entre os dois pontos. O kernel vale 1 para um ponto consigo mesmo e cai para perto de zero quando os dois se afastam: ele mede a semelhança pela distância.

O que o kernel polinomial calcula? Com $d = 2$ e duas coordenadas, abrir o quadrado de $1 + x_1 x'_1 + x_2 x'_2$ dá seis termos, e cada um é o produto de uma coluna calculada em $x$ pela mesma coluna calculada em $x'$. As colunas são $1$, $\sqrt{2}\,x_1$, $\sqrt{2}\,x_2$, $x_1^2$, $\sqrt{2}\,x_1 x_2$ e $x_2^2$, e a função `phi` as monta. O kernel e o produto interno nessas colunas, primeiro para $a$ e $b$, depois para todos os pares dos 200 pontos de treino:

In [ ]:
def phi(A):
    x1 = A[:, 0]
    x2 = A[:, 1]
    r = np.sqrt(2)
    return np.column_stack([
        np.ones(len(A)), r * x1,
        r * x2, x1**2,
        r * x1 * x2, x2**2])

print("(1 + <a, b>)² =",
    (1 + a @ b) ** 2)
fa, fb = phi(np.vstack([a, b]))
print("<φ(a), φ(b)> =",
    numero_br(fa @ fb, 4))
K2 = (1 + X @ X.T) ** 2
F = phi(X)
print("todos os pares iguais:",
    np.allclose(K2, F @ F.T))

> **🔧 Função**
>
> - **`np.ones(k)`** — um array com `k` uns: a coluna constante. **`np.sqrt(2)`** — a raiz quadrada, $\sqrt{2}$.
> - **`fa, fb = phi(...)`** — `phi` devolve duas linhas, uma por ponto, e cada uma vai para um nome.
> - **`X.T`** — a transposta: as linhas viram colunas. `X @ X.T` é a matriz dos produtos internos entre cada par de linhas de `X`, 200 por 200.

O kernel polinomial de grau 2 dá o mesmo número que o produto interno nessas seis colunas, sem calculá-las. Usar esse kernel equivale a ajustar o classificador de vetores de suporte num espaço com os termos de grau até 2, cada um multiplicado por uma constante fixa, $\sqrt{2}$ ou 1; com grau $d$, os termos vão até o grau $d$.

As colunas do `PolynomialFeatures` são as mesmas sem essas constantes e sem a coluna de uns. Por isso o kernel dá uma fronteira do mesmo tipo que a da figura acima, mas não a mesma. Em quantos pontos de teste as previsões do kernel de grau 2 e as do `ampliado` diferem?

In [ ]:
poli2 = SVC(kernel="poly",
    degree=2, gamma=1, coef0=1,
    C=1).fit(X, y)
p_k2 = poli2.predict(X_teste)
difere = (p_k2 != p_amp).mean()
print("previsões diferentes:",
    numero_br(difere, 4))

> **🔧 Função**
>
> - **`SVC(kernel="poly", degree=2, gamma=1, coef0=1)`** — a máquina de vetores de suporte com o kernel polinomial: o `SVC` multiplica o produto interno por `gamma`, soma `coef0` e eleva a `degree`. Com `gamma=1` e `coef0=1`, é a fórmula do kernel polinomial acima, com $d$ igual a `degree`.

As previsões diferem numa fração de 0,0048 dos pontos de teste. As constantes $\sqrt{2}$ põem as colunas em escalas diferentes das do `PolynomialFeatures`, e o mesmo `C`, que limita o tamanho dos coeficientes, pesa de um jeito diferente em cada uma; a coluna de uns também ganha um coeficiente limitado pelo `C`, ao contrário do intercepto.

Por que não montar as colunas, como no começo da seção? Com o kernel, o ajuste precisa de $K(x_i, x_{i'})$ para cada par de pontos de treino, e esse número depende de $n$, não do grau. As colunas do espaço ampliado dependem de $p$ e do grau. Num conjunto com 16 colunas e 207 linhas:

In [ ]:
from math import comb

p = 16
print("colunas, p = 16:")
cols = []
for d in [2, 3, 5, 10]:
    col = comb(p + d, d) - 1
    cols.append(col)
    print(f"  grau {d}:",
        numero_br(col, 0))
print("mais colunas que p:",
    min(cols) > p)
print("pares de pontos:")
for n in [207, 2_070, 20_700]:
    rot = numero_br(n, 0)
    pares = comb(n, 2)
    print(f"  n = {rot}:",
        numero_br(pares, 0))

> **🔧 Função**
>
> - **`comb(n, k)`**, do módulo `math` — de quantas formas se escolhem `k` itens entre `n`, sem importar a ordem: `comb(n, 2)` é o número de pares. `comb(p + d, d) - 1` conta os termos de grau 1 até $d$ em $p$ variáveis, sem o termo constante.

O critério é o custo de cada par. No espaço do kernel, que tem essas colunas multiplicadas pelas constantes e mais a coluna de uns, o produto interno de um par soma um produto por coluna. O kernel polinomial chega ao mesmo número com a soma dos 16 produtos de $\langle x_i, x_{i'} \rangle$: soma 1 e eleva à potência, qualquer que seja o grau. As colunas vão de 152 no grau 2 a 20.348 no grau 5 e 5.311.734 no grau 10, e já no grau 2 passam dos 16 produtos do kernel.

O número de pares não depende do grau: com 207 linhas, são 21.321. Ele cresce com o quadrado de $n$, e com 20.700 linhas seriam 214.234.650. O kernel compensa quando o grau é alto e $n$ não é enorme. O kernel radial é definido direto como função de dois pontos: não há colunas a montar.

Num ponto novo $x$, os vetores de suporte distantes de $x$ quase não entram na soma de $f(x)$ com o kernel radial; é a vizinhança de $x$ que decide a classe.

O `SVC` faz essas contas no `fit` e no `predict`; refazer a soma à mão serve só para ver a fórmula funcionando. Um `SVC` com kernel radial e $\gamma = 1$, e $f(x)$ refeito para 50 pontos de teste. A matriz `K` tem uma linha por ponto e uma coluna por vetor de suporte, e o produto `K @ alfa` faz, em cada linha, a soma $\sum_{i \in S} \alpha_i K(x, x_i)$:

In [ ]:
from sklearn.metrics.pairwise \
    import rbf_kernel

radial = SVC(kernel="rbf",
    gamma=1, C=1)
radial.fit(X, y)
x_novos = X_teste[:50]
sv = radial.support_vectors_
K = rbf_kernel(x_novos, sv,
    gamma=1)
print("K:", K.shape)
d2 = ((x_novos[0] - sv[0])
    ** 2).sum()
print("K[0, 0] = exp(−d²):",
    np.isclose(K[0, 0],
        np.exp(-1 * d2)))
alfa = radial.dual_coef_[0]
b0 = radial.intercept_[0]
f_mao = K @ alfa + b0
f_svc = radial.decision_function(
    x_novos)
print("igual ao SVC:",
    np.allclose(f_mao, f_svc))
print("β0 < 0:", b0 < 0)

> **🔧 Função**
>
> - **`SVC(kernel="rbf", gamma=1, C=1)`** — a máquina de vetores de suporte com o kernel radial; `"rbf"` vem de *radial basis function*, e `gamma` é o $\gamma$.
> - **`rbf_kernel(A, B, gamma)`**, de `sklearn.metrics.pairwise` — o kernel radial entre cada linha de `A` e cada linha de `B`, numa matriz com uma linha para cada linha de `A`.
> - **`K.shape`** — o número de linhas e o de colunas do array. É um atributo, e não leva parênteses.

A matriz `K` tem 50 linhas e 86 colunas, uma por vetor de suporte, e a soma refeita à mão coincide com o `decision_function`. O mesmo com o kernel polinomial de grau 3:

In [ ]:
from sklearn.metrics.pairwise \
    import polynomial_kernel

poli = SVC(kernel="poly",
    degree=3, gamma=1, coef0=1,
    C=1)
poli.fit(X, y)
sv_p = poli.support_vectors_
K_p = polynomial_kernel(x_novos,
    sv_p, degree=3, gamma=1,
    coef0=1)
formula = (
    1 + x_novos @ sv_p.T) ** 3
print("K_p = (1 + <x, x_i>)³:",
    np.allclose(K_p, formula))
alfa_p = poli.dual_coef_[0]
f_mao_p = (K_p @ alfa_p
    + poli.intercept_[0])
f_svc_p = poli.decision_function(
    x_novos)
print("igual ao SVC:",
    np.allclose(f_mao_p, f_svc_p))

> **🔧 Função**
>
> - **`polynomial_kernel(A, B, degree, gamma, coef0)`**, de `sklearn.metrics.pairwise` — o kernel polinomial do `SVC` entre cada linha de `A` e cada linha de `B`.

As duas máquinas, com os mesmos pontos:

In [ ]:
# Figura: Duas máquinas de vetores de suporte nos 200 pontos de treino, com `C=1`. Em cima, o kernel polinomial de grau 3 com `gamma=1` e `coef0=1`; embaixo, o kernel radial com $\gamma = 1$. A curva cinza contínua é a fronteira $f(x) = 0$, e as tracejadas, as bordas da margem, $f(x) = \pm 1$. No título, a taxa de erro nos 10.000 pontos de teste.
fig, eixos = plt.subplots(2, 1,
    figsize=(3.8, 6.6),
    sharex=True,
    layout="constrained")
for ax, m, nome in [
        (eixos[0], poli,
            "polinomial, grau 3"),
        (eixos[1], radial,
            "radial, γ = 1")]:
    regioes(ax, m)
    classes(ax, X, y)
    pm = m.predict(X_teste)
    em = (pm != y_teste).mean()
    ax.set_title(nome + " · erro "
        + numero_br(em, 3),
        fontsize=9)
eixos[1].set_xlabel("$X_1$")
fig.legend(handles, rotulos,
    ncols=2,
    loc="outside lower center")
plt.show()

As duas fronteiras cercam a nuvem do meio, mas diferem longe dela. O polinomial pinta de laranja o canto de cima à esquerda: acima dos poucos triângulos perto da borda de baixo dessa região, o canto só tem um triângulo isolado. Um polinômio de grau 3 continua crescendo à medida que se afasta dos dados, e o sinal que ele toma ali não vem das observações de treino próximas. O radial, longe de todos os vetores de suporte, fica com $f(x)$ perto de $\beta_0$, que é negativo, e dá a classe $-1$; a região laranja dele fica só perto dos triângulos.

### O $\gamma$ e o `C` do radial

No kernel radial, $\gamma$ diz quão depressa a semelhança cai com a distância. Com $\gamma$ grande, só os vetores de suporte muito próximos de $x$ pesam em $f(x)$, e a fronteira pode contornar pontos isolados. Antes da tabela, um palpite: quando $\gamma$ cresce, o erro de treino sobe ou desce? E o de teste?

In [ ]:
import pandas as pd

def mede(gama, c=1):
    m = SVC(kernel="rbf",
        gamma=gama, C=c)
    m.fit(X, y)
    ptr = m.predict(X)
    pte = m.predict(X_teste)
    e_tr = (ptr != y).mean()
    e_te = (pte != y_teste).mean()
    return {"gama": gama,
        "treino": e_tr,
        "teste": e_te,
        "n_vs": len(m.support_)}

gamas = [0.01, 0.1, 1, 10, 100]
tabela = pd.DataFrame(
    [mede(gm) for gm in gamas])
tabela.style.format(
    {"gama": "{:g}",
        "treino": "{:.3f}",
        "teste": "{:.4f}"},
    decimal=",").hide()

A coluna `n_vs` é o número de vetores de suporte. Dois valores de $\gamma$ da tabela, desenhados:

In [ ]:
# Figura: O kernel radial com `C=1` e dois valores de $\gamma$. Em cima, $\gamma = 0{,}1$; embaixo, $\gamma = 10$. A curva cinza contínua é a fronteira, e as tracejadas, as bordas da margem.
fig, eixos = plt.subplots(2, 1,
    figsize=(3.8, 6.6),
    sharex=True,
    layout="constrained")
for ax, gm in zip(eixos,
        [0.1, 10]):
    m = SVC(kernel="rbf",
        gamma=gm, C=1).fit(X, y)
    regioes(ax, m)
    classes(ax, X, y)
    rot = numero_br(gm, 1).rstrip(
        "0").rstrip(",")
    ax.set_title(f"γ = {rot}",
        fontsize=9)
eixos[1].set_xlabel("$X_1$")
fig.legend(handles, rotulos,
    ncols=2,
    loc="outside lower center")
plt.show()

Com $\gamma = 0{,}1$, a fronteira é uma curva lisa em volta da nuvem do meio. Com $\gamma = 10$, ela se quebra em ilhas em volta de grupos pequenos de pontos, e a margem contorna os círculos quase um por um. Três leituras da tabela, e as classes previstas com $\gamma = 0{,}01$:

In [ ]:
dif = np.diff(tabela["treino"])
print("treino não sobe:",
    np.all(dif <= 0))
i_min = tabela["teste"].idxmin()
g_min = tabela.loc[i_min, "gama"]
print("menor erro de teste: γ =",
    f"{g_min:g}".replace(
        ".", ","))
depois = tabela["teste"][i_min:]
print("teste sobe depois dele:",
    np.all(np.diff(depois) > 0))
m_01 = SVC(kernel="rbf",
    gamma=0.01).fit(X, y)
print("classes com γ = 0,01:",
    np.unique(m_01.predict(
        X_teste)))

> **🔧 Função**
>
> - **`tabela["teste"].idxmin()`** — o rótulo da linha com o menor valor da coluna; **`tabela.loc[i, "gama"]`**, o valor de `gama` nessa linha. **`tabela["teste"][i_min:]`** — a coluna da linha `i_min` em diante.

O erro de treino não sobe com $\gamma$: de 0,25 em $\gamma = 0{,}01$ vai a zero em $\gamma = 100$. O erro de teste desce e volta a subir. O menor é o de $\gamma = 0{,}1$, 0,1054, e em $\gamma = 100$ o erro de teste chega a 0,2364, perto dos 0,25 de prever $-1$ para todos. Com $\gamma = 100$, os 200 pontos de treino são vetores de suporte.

Em $\gamma = 0{,}01$, o kernel quase não distingue pontos próximos de pontos distantes: o ajuste prevê $-1$ para todos os pontos de teste, e os erros de treino e de teste ficam em 0,25.

O `C` do `SVC` também regula a flexibilidade. Na [seção 16.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/02-o-classificador-de-vetores-de-suporte.html), um `C` grande dá uma margem estreita, e menos pontos decidem a fronteira. Com $\gamma = 1$ e dois valores de `C`:

In [ ]:
for c in [1, 1e5]:
    r = mede(1, c=c)
    print(f"C = {c:,.0f}:"
        .replace(",", "."))
    print("  treino:",
        numero_br(r["treino"], 3))
    print("  teste:",
        numero_br(r["teste"], 4))

Com $\gamma = 1$, subir o `C` de 1 para 100.000 zera o erro de treino e leva o de teste de 0,1130 para 0,1768: com a margem estreita, a fronteira segue mais os pontos de treino.

### Escolher $\gamma$ e `C`

Com $\gamma$ e `C` para escolher, a validação cruzada da [seção 10.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/05-validacao-cruzada-em-classificacao.html) percorre a grade das combinações: 5 valores de `C` e 5 de $\gamma$, 25 pares, cada um avaliado nos mesmos 5 grupos. A grade leva as chaves `"C"` e `"gamma"`, e o `GridSearchCV` testa todas as combinações. A tabela mostra o erro de validação cruzada de cada par, com uma linha por `C` e uma coluna por $\gamma$:

In [ ]:
from sklearn.model_selection \
    import GridSearchCV
from sklearn.model_selection \
    import StratifiedKFold

grupos = StratifiedKFold(5,
    shuffle=True, random_state=16)
grade = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": gamas}
busca = GridSearchCV(
    SVC(kernel="rbf"), grade,
    cv=grupos)
busca.fit(X, y)
r = busca.cv_results_
acerto = r["mean_test_score"]
res = pd.DataFrame({
    "C": r["param_C"],
    "gama": r["param_gamma"],
    "erro": 1 - acerto})
res = res.astype(float)
(res.pivot(index="C",
        columns="gama",
        values="erro")
    .style.format("{:.3f}",
        decimal=",")
    .format_index("{:g}", axis=0,
        decimal=",")
    .format_index("{:g}", axis=1,
        decimal=","))

> **🔧 Função**
>
> - **`res.pivot(index="C", columns="gama", values="erro")`** — uma tabela com uma linha por valor de `C`, uma coluna por valor de `gama` e o `erro` de cada par nas células.
> - **`.format_index("{:g}", axis=0, decimal=",")`** — formata os rótulos das linhas (`axis=0`) ou das colunas (`axis=1`), sem mudar os valores.

A coluna de $\gamma = 100$, os três melhores pares e o escolhido:

In [ ]:
g100 = res[res["gama"] == 100]
e100 = g100["erro"]
print("γ = 100, todo C em 0,25:",
    np.allclose(e100, 0.25))
top = res.sort_values(
    "erro").head(3)
for _, lin in top.iterrows():
    print(f"C = {lin['C']:g},",
        f"γ = {lin['gama']:g}:"
        .replace(".", ","),
        numero_br(lin["erro"], 3))
empate = np.isclose(acerto,
    acerto.max()).sum()
print("pares no menor erro:",
    empate)
melhor = busca.best_params_
print("escolhido: C =",
    f"{melhor['C']:g}, γ =",
    f"{melhor['gamma']:g}"
    .replace(".", ","))
pb = busca.predict(X_teste)
print("erro de teste:", numero_br(
    (pb != y_teste).mean(), 4))

> **🔧 Função**
>
> - **`res[res["gama"] == 100]`** — as linhas de `res` em que a comparação dá `True`: aqui, as de `gama` igual a 100.
> - **`res.sort_values("erro")`** — as linhas em ordem crescente de `erro`; **`.head(3)`**, as três primeiras. **`.iterrows()`** percorre as linhas, uma por vez, como pares (rótulo, linha).

Com $\gamma = 100$, o erro de validação cruzada fica em 0,25 com qualquer `C`: a mesma taxa de uma regra que previsse $-1$ para todos os pontos.

A busca escolhe `C=1` e $\gamma = 0{,}1$, com erro de 0,085; os dois pares seguintes empatam em 0,100, e a diferença para eles é $0{,}100 - 0{,}085 = 0{,}015$. Nos 10.000 pontos de teste, o ajuste escolhido erra 0,1054.

O `np.isclose` conta quantos pares têm o acerto médio igual ao maior, a menos do arredondamento; com um empate exato, o `GridSearchCV` fica com o primeiro na grade. Aqui o par do menor erro é um só.

### Quanto dá para errar menos?

Os pontos foram simulados, e por isso o **classificador de Bayes** da [seção 7.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/07-classificacao-e-o-classificador-de-bayes.html), a regra de menor taxa de erro esperada, é conhecido. A classe $1$ é um quarto dos pontos, com densidade normal centrada na origem; a classe $-1$ é a mistura de metade dos pontos em torno de $(2; 2)$ e um quarto em torno de $(-2; -2)$. A regra de Bayes escolhe a classe com o maior produto de proporção e densidade. A constante das normais é a mesma nas três e se cancela:

In [ ]:
def normal(pts, centro):
    d2 = ((pts - centro)
        ** 2).sum(axis=1)
    return np.exp(-d2 / 2)

meio = 0.25 * normal(X_teste,
    [0, 0])
lados = (0.5 * normal(X_teste,
        [2, 2])
    + 0.25 * normal(X_teste,
        [-2, -2]))
regra = np.where(meio > lados,
    1, -1)
bayes = (regra != y_teste).mean()
print("regra de Bayes, teste:",
    numero_br(bayes, 4))

Nos mesmos 10.000 pontos, a regra de Bayes erra 0,0942, e o `SVC` escolhido, 0,1054: a diferença, $0{,}1054 - 0{,}0942 = 0{,}0112$, é o que o ajuste com 200 pontos de treino deixa de acertar em relação à regra que conhece as distribuições.

### Os padrões do `SVC`

De onde veio o $\gamma$ que o `SVC` usa quando não damos nenhum? Sem o argumento `gamma`, ele usa `gamma="scale"`, que calcula no `fit`, com os dados de treino,

$$
\gamma = \frac{1}{p \cdot v},
$$

com $p$ o número de colunas e $v$ a variância de todos os números de `X` juntos:

In [ ]:
p_x = X.shape[1]
gama = 1 / (p_x * X.var())
print("X.var():",
    numero_br(X.var(), 4))
print("1 / (p · X.var()):",
    numero_br(gama, 4))
padrao = SVC(kernel="rbf")
padrao.fit(X, y)
com_num = SVC(kernel="rbf",
    gamma=gama).fit(X, y)
iguais = np.allclose(
    padrao.decision_function(X),
    com_num.decision_function(X))
print("gamma='scale' = esse γ:",
    iguais)

> **🔧 Função**
>
> - **`X.var()`** — a variância de todos os números de `X` juntos, sem separar por coluna, com divisor igual à quantidade de números, $n \cdot p$.

O $\gamma$ que o `SVC` usa sem `gamma` depende dos dados: aqui, 0,1308. Com outros dados, ou com os mesmos dados numa outra unidade, sai outro número. No kernel polinomial, o padrão é `coef0=0`, e ele muda o resultado. Os dois `coef0`, com o mesmo $\gamma$ de `"scale"`:

In [ ]:
ajustes = {
    "poly padrão":
        SVC(kernel="poly"),
    "poly, coef0=1":
        SVC(kernel="poly",
            coef0=1),
    "rbf padrão": padrao}
for nome, m in ajustes.items():
    m.fit(X, y)
    pm = m.predict(X_teste)
    em = (pm != y_teste).mean()
    print(nome + ":",
        np.unique(pm), numero_br(
            em, 4))

Com `coef0=0` e grau 3, o kernel é $K(x, x_i) = (\gamma \langle x, x_i \rangle)^3$, e $f(x) - \beta_0$ só tem termos de grau 3. Uma função assim é ímpar: em $-x$, vale o oposto do que vale em $x$. Para qualquer ponto $x_0$, $f(x_0) - \beta_0 = -\big(f(-x_0) - \beta_0\big)$: o valor em $(-2; -2)$ é o oposto do valor em $(2; 2)$.

As duas nuvens da classe $-1$ ficam em lados opostos da origem, perto de $(2; 2)$ e de $(-2; -2)$, e por isso $f(x) - \beta_0$ tem sinais opostos nelas. Só um $\beta_0$ negativo o bastante para vencer a parte positiva põe as duas do lado $-1$, e aí a nuvem do meio vai junto: o ajuste prevê $-1$ para todos, com erro de 0,25, como o linear.

Com `coef0=1`, o $1 +$ da fórmula traz os termos de grau 0, 1 e 2, e o erro de teste cai para 0,1055. O radial com o $\gamma$ de `"scale"` erra 0,1053.

### Os pacientes do `Heart` e a curva ROC

O que o $\gamma$ faz no simulado, acertar o treino e errar os pontos novos quando cresce demais, se repete com pacientes? O conjunto `Heart` da [seção 12.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/03-arvores-de-classificacao.html) tem 13 medidas de pacientes com dor no peito e a resposta `doenca_cardiaca`, `"sim"` ou `"não"`. Sem as linhas com dado faltando, as colunas de texto viram indicadoras, e 90 pacientes ficam separados para teste, com a mesma proporção de doentes do conjunto todo:

In [ ]:
from sklearn.model_selection \
    import train_test_split

coracao = pd.read_csv(
    "dados/Heart.csv").dropna()
alvo = "doenca_cardiaca"
y_h = coracao[alvo]
X_h = pd.get_dummies(
    coracao.drop(columns=alvo),
    drop_first=True, dtype=float)
X_tr, X_te, y_tr, y_te = (
    train_test_split(X_h, y_h,
        test_size=90,
        random_state=16,
        stratify=y_h))
print("preditores:",
    coracao.shape[1] - 1)
print("linhas, colunas:",
    X_h.shape)
print("treino, teste:",
    len(X_tr), len(X_te))
print("treino:", y_tr
    .value_counts().to_dict())
print("teste:", y_te
    .value_counts().to_dict())

> **🔧 Função**
>
> - **`pd.read_csv("dados/Heart.csv").dropna()`** — lê a tabela e descarta as linhas com algum valor faltando. **`coracao.drop(columns=alvo)`** — a tabela sem a coluna `alvo`; a original não muda.
> - **`pd.get_dummies(tabela, drop_first=True, dtype=float)`** — troca cada coluna de texto por colunas indicadoras, de 0 e 1, uma por categoria; `drop_first=True` descarta a primeira categoria de cada coluna em ordem alfabética, que fica como base.
> - **`train_test_split(X, y, test_size=90, random_state=16, stratify=y)`**, de `sklearn.model_selection` — separa 90 linhas para teste, sorteadas com a semente 16; `stratify=y` mantém nos dois conjuntos a proporção de cada classe de `y`. **`.to_dict()`** — a contagem como um dicionário.

São 297 pacientes e 16 colunas, os 13 preditores com as indicadoras: 207 de treino e 90 de teste. As colunas estão em unidades diferentes (anos, mm Hg, mg/dL, indicadoras de 0 e 1), e o kernel radial mede distâncias, em que a coluna de maior variação pesa mais. Por isso cada modelo entra num `Pipeline` com o `StandardScaler`, que aprende as médias e os desvios só no treino ([seção 10.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/06-vazamento.html)).

Os modelos são o classificador de vetores de suporte, a análise discriminante linear (LDA) da [seção 9.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/07-leitura-complementar-lda-e-qda.html) e o kernel radial com três valores de $\gamma$; os `SVC`, todos com `C=1`. A LDA entra como uma segunda fronteira linear, já conhecida, para comparar. O escore dela não depende da escala dos preditores; ela entra no `Pipeline` só para os cinco modelos seguirem o mesmo caminho:

In [ ]:
from sklearn.preprocessing \
    import StandardScaler
from sklearn import (
    discriminant_analysis as da)

def com_escala(modelo):
    return make_pipeline(
        StandardScaler(), modelo)

LDA = (da
    .LinearDiscriminantAnalysis)
svc_h = SVC(kernel="linear", C=1)
modelos = {
    "SVC linear":
        com_escala(svc_h),
    "LDA": com_escala(LDA())}
for gm in [0.001, 0.01, 0.1]:
    nome = f"γ = {gm:g}".replace(
        ".", ",")
    rbf = SVC(kernel="rbf",
        gamma=gm, C=1)
    modelos[nome] = com_escala(
        rbf)

> **🔧 Função**
>
> - **`StandardScaler()`**, de `sklearn.preprocessing` — subtrai de cada coluna a sua média e divide pelo seu desvio padrão, ambos calculados no `fit`, só com as linhas de treino.
> - **`da.LinearDiscriminantAnalysis()`**, do módulo `sklearn.discriminant_analysis` — a LDA; o `decision_function` dela também é um escore que cresce com a chance da classe `"sim"`. O código guarda a classe no nome `LDA` e cria um modelo com `LDA()`.
> - **`m.decision_function(X_te)`** num `Pipeline` — passa as linhas pelo `StandardScaler` e entrega o resultado ao modelo.

Para cada um, o código calcula o escore $f(x)$ do `decision_function` nos dois conjuntos e a área sob a curva ROC (AUC, de *area under the curve*) da [seção 9.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/05-avaliando-um-classificador.html):

In [ ]:
from sklearn.metrics import (
    roc_auc_score)

escores = {}
linhas = []
for nome, m in modelos.items():
    m.fit(X_tr, y_tr)
    s_tr = m.decision_function(
        X_tr)
    s_te = m.decision_function(
        X_te)
    escores[nome] = (s_tr, s_te)
    p_te = m.predict(X_te)
    a_tr = roc_auc_score(
        y_tr, s_tr)
    a_te = roc_auc_score(
        y_te, s_te)
    linhas.append({"modelo": nome,
        "auc_treino": a_tr,
        "auc_teste": a_te,
        "erros_teste":
            (p_te != y_te).sum()})
m_lin = modelos["SVC linear"]
print("classes_:", m_lin.classes_)
auc = pd.DataFrame(linhas)
auc.style.format(
    {"auc_treino": "{:.3f}",
        "auc_teste": "{:.3f}"},
    decimal=",").hide()

> **🔧 Função**
>
> - **`m.classes_`** num `Pipeline` — as classes do último passo: escore positivo quer dizer `classes_[1]`, `"sim"`.
> - **`roc_auc_score(y, escore)`**, de `sklearn.metrics` — a área sob a curva ROC; com rótulos de texto, a classe positiva é a última em ordem alfabética, `"sim"`.

Com `"sim"` em `classes_[1]`, um escore maior quer dizer mais indício de doença, e é esse o sentido de que a curva ROC precisa. As leituras da tabela, uma a uma:

In [ ]:
radiais = auc.iloc[2:]
sobe = np.all(np.diff(
    radiais["auc_treino"]) > 0)
print("AUC de treino sobe com γ:",
    sobe)
i_tr = auc["auc_treino"].idxmax()
print("maior AUC de treino:",
    auc.loc[i_tr, "modelo"])
i_te = auc["auc_teste"].idxmin()
print("menor AUC de teste:",
    auc.loc[i_te, "modelo"])
i_er = auc["erros_teste"].idxmax()
print("mais erros de teste:",
    auc.loc[i_er, "modelo"])
dif = (auc.iloc[0, 1:3]
    - auc.iloc[1, 1:3]).abs()
print("|SVC − LDA| < 0,01:",
    np.all(dif < 0.01))

> **🔧 Função**
>
> - **`auc["auc_treino"].idxmax()`** — o rótulo da linha com o maior valor da coluna, como o `idxmin` faz com o menor. **`.abs()`** — o valor absoluto de cada elemento da série.

No treino, a área sob a curva sobe com $\gamma$, de 0,906 para 0,920 e 0,990, e o $\gamma = 0{,}1$ tem a maior área de treino dos cinco. No teste, ele tem a menor área, 0,895, e o maior número de erros, 17 dos 90 pacientes. São 90 pacientes num só sorteio, e a diferença para os outros modelos é de poucos pacientes. Na tabela do simulado, o $\gamma$ fez o mesmo: acima de certo valor, o treino não piora e o teste piora.

As duas fronteiras lineares, o classificador de vetores de suporte e a LDA, ficam a menos de 0,01 uma da outra na área, no treino e no teste; no teste, erram 15 e 13 pacientes.

As curvas ROC do classificador de vetores de suporte e do kernel radial com os três valores de $\gamma$, no treino e no teste:

In [ ]:
# Figura: Curvas ROC no `Heart`, em cima no treino e embaixo no teste: o classificador de vetores de suporte, em laranja, e o kernel radial com $\gamma = 0{,}001$ (azul), $0{,}01$ (verde) e $0{,}1$ (roxo), todos com `C=1`. No treino, a curva roxa encosta no canto de cima à esquerda; no teste, ela fica abaixo das outras enquanto a taxa de falso alarme é baixa.
from sklearn.metrics import (
    roc_curve)

def painel(ax, nomes, cores, k):
    yy = [y_tr, y_te][k]
    for nome, cor in zip(nomes,
            cores):
        s = escores[nome][k]
        fa, rev, _ = roc_curve(yy,
            s, pos_label="sim")
        rot = None
        if k == 0:
            rot = nome
        ax.plot(fa, rev, lw=1.8,
            color=cor, label=rot)
    ax.plot([0, 1], [0, 1],
        color="0.45", ls="--",
        lw=1)
    marcas = [0, 0.5, 1]
    textos = ["0", "0,5", "1"]
    ax.set_xticks(marcas, textos)
    ax.set_yticks(marcas, textos)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_aspect("equal")
    ax.set_ylabel("revocação")

embaixo = "outside lower center"

def rocs(nomes, cores):
    fig, eixos = plt.subplots(
        2, 1, figsize=(3.8, 6.6),
        sharex=True,
        layout="constrained")
    for k in [0, 1]:
        painel(eixos[k], nomes,
            cores, k)
    eixos[0].set_title("treino",
        fontsize=9)
    eixos[1].set_title("teste",
        fontsize=9)
    eixos[1].set_xlabel(
        "taxa de falso alarme")
    fig.legend(ncols=2,
        loc=embaixo)
    plt.show()

rocs(["SVC linear", "γ = 0,001",
        "γ = 0,01", "γ = 0,1"],
    ["C1", "C0", "C2", "C3"])

> **🔧 Função**
>
> - **`roc_curve(y, escore, pos_label="sim")`**, de `sklearn.metrics` — a taxa de falso alarme, a revocação e o limiar de cada ponto da curva ROC; `pos_label` diz qual rótulo é a classe positiva. O `_` descarta os limiares.
> - **`ax.set_xticks(marcas, textos)`** — o segundo argumento, `textos`, é o texto de cada marca, aqui com vírgula decimal.

No treino, a curva roxa, do $\gamma = 0{,}1$, fica acima das outras; no teste, ela fica abaixo delas nos limiares de pouco falso alarme. É a área de treino mais alta e a de teste mais baixa da tabela, vistas limiar a limiar.

### A escala dos preditores

Sem o `StandardScaler`, o que muda? E se o `colesterol` estiver em mmol/L em vez de mg/dL, dividido por 38,67? Antes de olhar, um palpite: a troca de unidade muda o resultado com a padronização? E sem ela? O kernel radial com o $\gamma$ de `"scale"`, nas quatro combinações, e os erros nos 90 pacientes de teste:

In [ ]:
def avalia(Xa, Xb, escala):
    m = SVC(kernel="rbf")
    if escala:
        m = com_escala(m)
    m.fit(Xa, y_tr)
    s = m.decision_function(Xb)
    pb = m.predict(Xb)
    e = (pb != y_te).sum()
    a = roc_auc_score(y_te, s)
    return e, a

X_tr2 = X_tr.copy()
X_te2 = X_te.copy()
for t in [X_tr2, X_te2]:
    t["colesterol"] /= 38.67
unid = [(X_tr, X_te, "mg/dL"),
    (X_tr2, X_te2, "mmol/L")]
quatro = []
for escala in [False, True]:
    rot = ["sem", "com"][escala]
    for Xa, Xb, un in unid:
        e, a = avalia(Xa, Xb,
            escala)
        quatro.append({
            "escala": rot,
            "unidade": un,
            "erros": e, "auc": a})
q = pd.DataFrame(quatro)
com = q[q["escala"] == "com"]
e_mg, e_mm = com["erros"]
a_mg, a_mm = com["auc"]
print("com escala, erros iguais:",
    e_mg == e_mm)
print("com escala, AUC iguais:",
    np.isclose(a_mg, a_mm))
q.style.format(
    {"auc": "{:.3f}"},
    decimal=",").hide()

> **🔧 Função**
>
> - **`t["colesterol"] /= 38.67`** — divide a coluna por 38,67, na própria tabela; as cópias feitas com `.copy()` deixam `X_tr` e `X_te` como estavam.
> - **`["sem", "com"][escala]`** — `False` vale 0 e `True` vale 1 como posição da lista.

Sem a padronização, o modelo erra 32 dos 90 pacientes, contra 16 com o `StandardScaler`. Por que tanto? O código procura a coluna de maior variância e o $\gamma$ que o `"scale"` calcula sem padronizar:

In [ ]:
variancias = X_tr.var(ddof=0)
print("maior variância:",
    variancias.idxmax())
v = X_tr.to_numpy().var()
print("γ sem escala:",
    f"{1 / (16 * v):.1e}"
    .replace(".", ","))

> **🔧 Função**
>
> - **`X_tr.var(ddof=0)`** — a variância de cada coluna, com divisor igual ao número de linhas. **`X_tr.to_numpy().var()`** — a variância de todos os números juntos, a que o `gamma="scale"` usa.

Sem a padronização, a coluna de maior variância é o `colesterol`, e o $\gamma$ de `"scale"`, calculado com a variância de todos os números da tabela juntos, sai perto de $10^{-5}$. Nas distâncias entre pacientes, as colunas de variância grande pesam mais que as indicadoras de 0 e 1.

Mudar só a unidade do `colesterol` muda o resultado sem padronização, para 30 erros. Com ela, nada muda: continuam 16 erros, e as duas áreas são iguais, como o `np.isclose` confere. A padronização põe cada coluna na mesma escala, qualquer que seja a unidade de origem.

### Escolhas e limites do kernel

O problema do classificador de vetores de suporte continua o mesmo; o que muda é a medida de semelhança entre dois pontos. O classificador depende dos pontos só pelos produtos internos, e trocar o produto interno por um kernel faz a conta num espaço com mais colunas sem montá-lo; no kernel polinomial, são os termos de grau até $d$, cada um multiplicado por uma constante. No plano original, o hiperplano desse espaço aparece como uma fronteira curva.

Em troca, há mais a escolher: o kernel, o $\gamma$ ou o grau, e o `C` do `SVC`, todos por validação cruzada e com os preditores na mesma escala. No simulado, a escolha chegou a 0,1054 de erro de teste, contra 0,0942 da regra de Bayes; no `Heart`, o kernel radial mais flexível teve a menor área e o maior número de erros neste teste.

O método também tem limites. O ajuste precisa do kernel para cada par de pontos de treino, e esse número cresce com o quadrado de $n$. O `SVC` devolve um escore $f(x)$ e uma classe, no padrão do `SVC`, não uma probabilidade. E, fora do kernel linear, não há um coeficiente por preditor para ler: a fronteira fica escrita como uma soma sobre os vetores de suporte.

## Mais de Duas Classes

> **📌 Nota**
>
> Esta seção corresponde às seções 9.4, 9.6.4 e 9.6.5 de James et al. (2023).

Um hiperplano divide o espaço em dois lados, e o classificador de vetores de suporte dá uma classe a cada lado. Uma amostra de tecido de um tumor infantil, porém, pode vir de quatro tipos diferentes, e dois lados não bastam. Como juntar vários classificadores de duas classes numa resposta só? E o que fazer quando eles discordam?

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("estilo-figuras.mplstyle")

Os números impressos seguem a vírgula decimal da prosa:

In [ ]:
numero_br = lambda v, casas=2: (
    f"{v:,.{casas}f}"
    .replace(",", "_")
    .replace(".", ",")
    .replace("_", ".")
    .replace("-", "−")
)

### Um contra um

A primeira resposta põe as classes para disputar duas a duas.

> **🔷 Conceito**
>
> Com $K$ classes, há $\binom{K}{2} = K(K-1)/2$ pares de classes. A abordagem **um contra um** (*one versus one*) ajusta um classificador de duas classes para cada par, só com as observações daquelas duas classes. Um ponto novo passa pelos $\binom{K}{2}$ classificadores, cada um dá o seu voto a uma das duas classes do seu par, e a previsão é a classe com mais votos.

Com 3 classes, são 3 classificadores; com 4, são 6.

Os dados são três nuvens de pontos no plano, nos vértices de um triângulo: a classe 0 embaixo à esquerda, a classe 1 embaixo à direita e a classe 2 em cima, misturadas no meio, onde as três se encontram. Os pontos de treino são 50 de cada classe, sorteados de normais com desvio padrão 1 nas duas coordenadas, em torno de três centros: $(0; 0)$ para a classe 0, $(3; 0)$ para a classe 1 e $(1{,}5; 2{,}6)$ para a classe 2. Depois deles, o mesmo gerador sorteia 3.000 pontos de teste por classe:

In [ ]:
import numpy as np

rng = np.random.default_rng(16)
centros = np.array(
    [[0, 0], [3, 0], [1.5, 2.6]])

def tres(n):
    classe = np.repeat(
        [0, 1, 2], n)
    ruido = rng.normal(
        size=(3 * n, 2))
    pts = centros[classe] + ruido
    return pts, classe

X, y = tres(50)
X_teste, y_teste = tres(3_000)
print("treino:", np.bincount(y))
print("teste:",
    np.bincount(y_teste))

> **🔧 Função**
>
> - **`centros[classe]`** — uma linha de `centros` para cada elemento de `classe`: o centro da classe de cada ponto, somado ao ruído na linha seguinte.
> - **`np.bincount(y)`** — quantas vezes aparece cada inteiro de 0 até o maior valor de `y`: aqui, quantos pontos há em cada classe.

Os pares, na ordem em que o `SVC` guarda os classificadores:

In [ ]:
from itertools import combinations

pares = list(combinations(
    range(3), 2))
print(pares)

> **🔧 Função**
>
> - **`combinations(range(3), 2)`**, do módulo `itertools` — todos os pares de elementos de `range(3)`, sem repetir e sem importar a ordem; `list` os reúne numa lista.

O `SVC` já ajusta um classificador por par quando `y` tem mais de duas classes. Com `decision_function_shape="ovo"` (o nome vem de *one versus one*), o `decision_function` devolve uma coluna por par, na ordem de `pares`, e o sinal de cada coluna diz qual classe do par leva o voto:

In [ ]:
from sklearn.svm import SVC

ovo = SVC(kernel="linear", C=1,
    decision_function_shape="ovo")
ovo.fit(X, y)
d = ovo.decision_function(
    X_teste)
print("d:", d.shape)

> **🔧 Função**
>
> - **`SVC(..., decision_function_shape="ovo")`** — o `decision_function` devolve uma coluna por par de classes. O padrão, `"ovr"`, devolve uma coluna por classe; o ajuste é o mesmo nos dois casos.

As regiões de cada classe e as retas dos três classificadores de pares:

In [ ]:
# Figura: O um contra um nos 150 pontos de treino, com `C` = 1: os círculos azuis são a classe 0, os triângulos laranja, a classe 1, e os quadrados verdes, a classe 2. O fundo tem a cor da classe prevista. As três retas cinza tracejadas são as fronteiras $f(x) = 0$ dos classificadores dos pares $(0, 1)$, $(0, 2)$ e $(1, 2)$.
import matplotlib.pyplot as plt

g = np.linspace(-3.5, 6.5, 301)
g1, g2 = np.meshgrid(g, g)
forma = g1.shape
malha = np.column_stack(
    [g1.ravel(), g2.ravel()])
cores = ["C0", "C1", "C2"]
marcas = ["o", "^", "s"]

def pontos(ax):
    for c in range(3):
        p = X[y == c]
        ax.scatter(
            p[:, 0], p[:, 1],
            color=cores[c], s=14,
            marker=marcas[c],
            label=f"classe {c}")
    ax.set_aspect("equal")
    ax.set_xlim(-3.5, 6.5)
    ax.set_ylim(-3.5, 6.5)
    marcas_eixo = [-2, 0, 2, 4, 6]
    ax.set_xticks(marcas_eixo)
    ax.set_yticks(marcas_eixo)
    ax.set_xlabel("$X_1$")
    ax.set_ylabel("$X_2$")

def regioes(ax, classe):
    z = classe.reshape(forma)
    faixas = [-0.5, 0.5, 1.5, 2.5]
    ax.contourf(g1, g2, z,
        levels=faixas,
        colors=cores, alpha=0.2)

fig, ax = plt.subplots(
    figsize=(3.8, 4.3),
    layout="constrained")
regioes(ax, ovo.predict(malha))
d_m = ovo.decision_function(malha)
for k in range(3):
    zk = d_m[:, k].reshape(forma)
    ax.contour(g1, g2, zk,
        levels=[0], colors="0.45",
        linewidths=1,
        linestyles="--")
pontos(ax)
fig.legend(ncols=3,
    loc="outside lower center")
plt.show()

Cada reta separa as duas classes do seu par, mas só vale como fronteira onde essas duas classes disputam o ponto. A reta do par $(0, 1)$, por exemplo, sobe entre a nuvem azul e a laranja; acima do centro, ela atravessa a região verde, onde a classe 2 já ganhou os seus dois pares e a disputa entre 0 e 1 não muda a previsão.

> **⚠️ Atenção — O sinal no um contra um**
>
> Com duas classes, $f(x)$ positivo quer dizer `classes_[1]`. Nas colunas do `"ovo"`, é o contrário: o valor positivo dá o voto à **primeira** classe do par.

O $f(x)$ do classificador do par $(0, 1)$, ajustado à parte só com as classes 0 e 1, é o oposto da primeira coluna:

In [ ]:
par = y < 2
so_01 = SVC(kernel="linear", C=1)
so_01.fit(X[par], y[par])
print("classes_:", so_01.classes_)
f_01 = so_01.decision_function(
    X_teste)
print("d[:, 0] = −f_01:",
    np.allclose(d[:, 0], -f_01))

A função `vota` só conta votos, e pode ser lida como uma caixa-preta: para cada par $(a, b)$, o voto vai para $a$ onde a coluna é positiva, e para $b$ onde não é.

In [ ]:
def vota(d, K):
    votos = np.zeros(
        (len(d), K), dtype=int)
    pares_k = combinations(
        range(K), 2)
    for k, (a, b) in enumerate(
            pares_k):
        a_ganha = d[:, k] > 0
        votos[a_ganha, a] += 1
        votos[~a_ganha, b] += 1
    return votos

votos = vota(d, 3)

> **🔧 Função**
>
> - **`np.zeros((n, K), dtype=int)`** — uma matriz de `n` linhas e `K` colunas, cheia de zeros inteiros: uma linha por ponto, uma coluna por classe.
> - **`votos[a_ganha, a] += 1`** — nas linhas em que `a_ganha` é `True`, soma 1 na coluna da classe `a`. O `~` troca `True` por `False` e vice-versa: `~a_ganha` marca as linhas em que o voto vai para `b`.

Cada linha de `votos` tem os votos das classes 0, 1 e 2. Os votos do ponto de teste mais perto de cada centro, e do mais perto do meio do triângulo, a média dos três centros:

In [ ]:
meio = centros.mean(axis=0)
alvos = np.vstack([centros, meio])
nomes = ["centro 0", "centro 1",
    "centro 2", "meio"]
for nome, c in zip(nomes, alvos):
    dist = np.linalg.norm(
        X_teste - c, axis=1)
    i = dist.argmin()
    print(f"{nome}: classe",
        y_teste[i], "votos",
        votos[i])

Com `axis=1`, a norma é calculada linha a linha, e `argmin` dá a posição do menor valor, o ponto mais perto.

Perto de cada centro, a classe do centro ganha os seus dois pares, e o terceiro classificador, que não a envolve, dá o voto que sobra a outra classe. O ponto do meio é da classe 2, mas a classe 0 ganha os dois pares de que participa: perto de onde as nuvens se encontram, os classificadores podem levar o voto para uma classe que não é a do ponto.

Com 3 classes, os 3 votos de um ponto se dividem em 2, 1 e 0, ou num empate de 1 a 1 a 1. A classe mais votada é sempre a previsão do `SVC`?

In [ ]:
prev = ovo.predict(X_teste)
mais = votos.argmax(axis=1)
print("mais votada = predict:",
    np.all(mais == prev))
empate = votos.max(axis=1) == 1
print("empates 1 a 1 a 1:",
    empate.sum())
e_ovo = (prev != y_teste).mean()
print("erro de teste:",
    numero_br(e_ovo, 4))

Nos 9.000 pontos de teste, a classe mais votada é a previsão do `SVC`, e nenhum tem empate. A taxa de erro de teste é 0,1141.

Um empate só acontece onde cada classe ganha um par, perto de onde as três retas quase se encontram. A malha da figura não tem nenhum nó empatado; uma malha bem mais fina, com passo de 0,001 em volta do meio do triângulo, encontra alguns:

In [ ]:
v_m = vota(d_m, 3)
print("empates na malha:",
    (v_m.max(axis=1) == 1).sum())
prev_m = ovo.predict(malha)
nos = np.arange(len(v_m))
print("2 votos em todo nó:",
    np.all(v_m[nos, prev_m] == 2))
z = np.linspace(-0.5, 0.5, 1001)
z1, z2 = np.meshgrid(
    z + meio[0], z + meio[1])
fina = np.column_stack(
    [z1.ravel(), z2.ravel()])
d_f = ovo.decision_function(fina)
v_f = vota(d_f, 3)
emp_f = v_f.max(axis=1) == 1
print("nós da malha fina:",
    numero_br(len(fina), 0))
print("empates:", emp_f.sum())
p_emp = ovo.predict(fina[emp_f])
print("predict nos empates:",
    np.unique(p_emp))

Em todos os nós da malha da figura, a classe prevista tem 2 votos: ganha os dois pares de que participa. O `v_m[nos, prev_m]` pega, em cada nó, os votos da classe prevista. Na malha fina, são 305 empates em 1.002.001 nós, e nos 305 o `predict` fica com a classe 0, a primeira das classes empatadas na ordem de `classes_`.

O `SVC` aceita `break_ties=True`, que só funciona com o padrão `"ovr"` e desempata pela coluna de maior valor do `decision_function`.

### Um contra todos

A outra resposta põe cada classe contra todas as outras juntas.

> **🔷 Conceito**
>
> A abordagem **um contra todos** (*one versus all*, também chamada de um contra o resto) ajusta $K$ classificadores. O $k$-ésimo trata a classe $k$ como $+1$ e todas as outras juntas como $-1$, e dá a cada ponto $x^*$ um escore
>
> $$
> f_k(x^*) = \beta_{0k} + \sum_{j=1}^{p} \beta_{jk} x^*_j.
> $$
>
> A previsão é a classe de maior escore.

Com $f_k(x^*)$ positivo, $x^*$ fica do lado da classe $k$ na fronteira do $k$-ésimo classificador; com negativo, do lado das outras. Quanto maior o $f_k(x^*)$, mais esse classificador aposta na classe $k$.

O `OneVsRestClassifier` monta os $K$ classificadores a partir de um `SVC` de duas classes. Quantos pontos de teste têm exatamente um $f_k$ positivo?

In [ ]:
from sklearn.multiclass import (
    OneVsRestClassifier)

ovr = OneVsRestClassifier(
    SVC(kernel="linear", C=1))
ovr.fit(X, y)
print("classificadores:",
    len(ovr.estimators_))
F = ovr.decision_function(X_teste)
print("F:", F.shape)
positivos = (F > 0).sum(axis=1)
print("nenhum f_k > 0:",
    (positivos == 0).sum())
print("um só:",
    (positivos == 1).sum())
print("dois:",
    (positivos == 2).sum())
print("três:",
    (positivos == 3).sum())
p_ovr = ovr.predict(X_teste)
maior = F.argmax(axis=1)
print("maior escore = predict:",
    np.all(maior == p_ovr))

> **🔧 Função**
>
> - **`OneVsRestClassifier(estimador)`**, de `sklearn.multiclass` — ajusta uma cópia do estimador para cada classe, com essa classe contra todas as outras. **`ovr.estimators_`** — a lista dos classificadores ajustados, um por classe.
> - **`ovr.decision_function(X)`** — o escore $f_k$ de cada classificador, uma coluna por classe; o `predict` dá a classe da coluna de maior escore.

Dos 9.000 pontos de teste, 7.550 têm um só $f_k$ positivo. Nos outros, a regra pelo sinal não decide: 855 não têm nenhum escore positivo, e nenhum classificador reclama o ponto para a sua classe; 595 têm dois, e dois classificadores reclamam o mesmo ponto. Nenhum tem os três escores positivos. O maior escore resolve os dois casos.

Antes de ver a figura: em que parte do plano devem ficar os pontos sem nenhum escore positivo? E os pontos com mais de um?

In [ ]:
# Figura: O um contra todos nos mesmos pontos, com `C` = 1. O fundo tem a cor da classe de maior escore. As retas tracejadas são as fronteiras $f_k(x) = 0$ de cada classificador, na cor da sua classe: de cada reta, a classe $k$ fica do lado da sua nuvem. Na zona cinza, no meio, nenhum dos três escores é positivo; nas faixas hachuradas, mais de um é.
F_m = ovr.decision_function(malha)
n_pos = (F_m > 0).sum(axis=1)
z_pos = n_pos.reshape(forma)
fig, ax = plt.subplots(
    figsize=(3.8, 4.3),
    layout="constrained")
regioes(ax, ovr.predict(malha))
ax.contourf(g1, g2, z_pos,
    levels=[-0.5, 0.5],
    colors=["0.6"], alpha=0.7)
ax.contour(g1, g2, z_pos,
    levels=[0.5], colors="0.45",
    linewidths=0.8)
plt.rcParams.update(
    {"hatch.color": "0.45"})
ax.contourf(g1, g2, z_pos,
    levels=[1.5, 3.5],
    colors="none",
    hatches=["////"])
for k in range(3):
    zk = F_m[:, k].reshape(forma)
    ax.contour(g1, g2, zk,
        levels=[0],
        colors=cores[k],
        linewidths=1.5,
        linestyles="--")
pontos(ax)
fig.legend(ncols=3,
    loc="outside lower center")
plt.show()

> **🔧 Função**
>
> - **`(F_m > 0).sum(axis=1)`** — em cada nó da malha, quantos dos três escores são positivos. Com `levels=[-0.5, 0.5]`, o `contourf` pinta só os nós em que essa conta é 0; com `levels=[1.5, 3.5]`, os nós em que ela é 2 ou 3.
> - **`hatches=["////"]`**, no `contourf` — hachuras diagonais na faixa; a cor delas vem de `"hatch.color"`, em `plt.rcParams`.

A zona cinza fica no meio, onde as três nuvens se encontram: ali, os três classificadores põem o ponto do lado de "todas as outras classes", e nenhum escore é positivo. As faixas hachuradas saem dos vértices da zona cinza para as bordas, cada uma entre duas nuvens: ali, mais de um classificador põe o ponto do lado da sua classe. Quantas previsões mudam de uma abordagem para a outra, e quantas dessas caem nos pontos em que o sinal não decide?

In [ ]:
e_ovr = (p_ovr != y_teste).mean()
print("erro, um contra um:",
    numero_br(e_ovo, 4))
print("erro, um contra todos:",
    numero_br(e_ovr, 4))
difere = prev != p_ovr
print("previsões diferentes:",
    difere.sum())
ambiguo = positivos != 1
print("  sem um só f_k > 0:",
    (difere & ambiguo).sum())

As duas abordagens discordam em 247 dos 9.000 pontos de teste, e 231 deles são pontos sem exatamente um escore positivo. As taxas de erro de teste ficam em 0,1141 e 0,1114, uma diferença de $0{,}1141 - 0{,}1114 = 0{,}0027$, cerca de 24 dos 9.000 pontos. É um único sorteio, com fronteiras lineares nos dois casos, e nenhuma das duas abordagens erra bem mais que a outra.

### O que o `SVC` guarda

Com o padrão, `decision_function_shape="ovr"`, o `decision_function` do `SVC` devolve uma coluna por classe. Seria esse o um contra todos? Não: o `SVC` ajusta sempre um classificador por par, e o `"ovr"` monta cada coluna a partir dos votos do um contra um, sem ajustar nenhum classificador um contra todos.

In [ ]:
padrao = SVC(kernel="linear", C=1)
padrao.fit(X, y)
print("coef_:",
    padrao.coef_.shape)
d_pad = padrao.decision_function(
    X_teste)
print("colunas, 'ovr':",
    d_pad.shape[1])
dif = np.abs(d_pad - votos)
print("|ovr − votos| < 1/3:",
    np.all(dif < 1 / 3))
p_pad = padrao.predict(X_teste)
print("previsões = ovo:",
    np.all(p_pad == prev))

O `coef_` tem uma linha por par, com os $\beta_1$ e $\beta_2$ de cada classificador: 3 linhas e 2 colunas. Cada coluna do `"ovr"` é, a menos de 1/3, o número de votos da classe, e as previsões são as mesmas do um contra um.

A diferença para os votos é um ajuste tirado dos valores de $f(x)$ dos pares, sempre menor que 1/3 de voto. Como os votos são inteiros, esse ajuste não muda qual classe tem mais votos; só decide num empate, e é ele que o `break_ties=True` usa. Com $K = 3$, as duas formas têm 3 colunas, porque há tantos pares quanto classes; os dados com 4 classes, a seguir, separam as duas.

### Quatro tipos de tumor

A tabela `Khan` tem a expressão de milhares de genes, medida em amostras de quatro tipos de tumor infantil: linfoma de Burkitt (LB), sarcoma de Ewing (SE), neuroblastoma (NB) e rabdomiossarcoma (RMS). A coluna `conjunto` guarda a divisão entre treino e teste feita pelos autores do estudo, e a coluna `tumor`, o tipo. O código troca os nomes dos tumores pelas siglas:

In [ ]:
import pandas as pd

khan = pd.read_csv(
    "dados/Khan.csv.gz")
conj = khan["conjunto"]
treino = conj == "treino"
genes = khan.filter(like="gene_")
siglas = {
    "linfoma de Burkitt": "LB",
    "sarcoma de Ewing": "SE",
    "neuroblastoma": "NB",
    "rabdomiossarcoma": "RMS"}
tumor = khan["tumor"].map(siglas)
X_tr = genes[treino]
X_te = genes[~treino]
y_tr = tumor[treino]
y_te = tumor[~treino]
print("treino:", X_tr.shape)
print("teste:", X_te.shape)
dp = X_tr.std()
print("desvio dos genes:",
    numero_br(dp.min()), "a",
    numero_br(dp.max()))
pd.crosstab(tumor, conj)

> **🔧 Função**
>
> - **`khan.filter(like="gene_")`** — as colunas cujo nome contém `"gene_"`: os 2.308 genes, sem `conjunto` e sem `tumor`.
> - **`khan["tumor"].map(siglas)`** — troca cada valor da coluna pelo valor correspondente no dicionário `siglas`.
> - **`genes[~treino]`** — as linhas em que `treino` é `False`.
> - **`X_tr.std()`** — o desvio padrão de cada coluna.
> - **`pd.crosstab(a, b)`** — conta quantas linhas há em cada combinação de um valor de `a`, nas linhas da tabela, com um valor de `b`, nas colunas.

São 63 amostras de treino e 20 de teste, $63 + 20 = 83$ ao todo, e 2.308 colunas: $p$ é muito maior que $n$. Com mais colunas que amostras, um ajuste linear já tem flexibilidade de sobra, como os mínimos quadrados da [seção 11.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/07-o-que-muda-em-alta-dimensao.html), e o kernel de partida é o linear.

Os genes entram sem padronizar: são todos medidas de expressão, e padronizar daria a cada gene o mesmo peso, inclusive aos que variam pouco. No treino, o desvio padrão dos genes vai de 0,27 a 1,92. Os tumores são quatro, e o `SVC` ajusta os 6 pares:

In [ ]:
svc_k = SVC(kernel="linear", C=10)
svc_k.fit(X_tr, y_tr)
print("classes_:", svc_k.classes_)
p_tr = svc_k.predict(X_tr)
p_te = svc_k.predict(X_te)
print("erros de treino:",
    (p_tr != y_tr).sum())
print("erros de teste:",
    (p_te != y_te).sum())
pd.crosstab(y_te.to_numpy(), p_te,
    rownames=["real"],
    colnames=["prevista"])

> **🔧 Função**
>
> - **`pd.crosstab(..., rownames, colnames)`** — `rownames` e `colnames` dão o nome das linhas e das colunas da tabela. Com **`y_te.to_numpy()`**, sem o rótulo das linhas, como o `p_te`, o `crosstab` junta os dois pela posição.

O ajuste não erra nenhuma das 63 amostras de treino. Nas 20 de teste, erra 2: as duas são neuroblastomas previstos como sarcoma de Ewing. Padronizar os genes, ou trocar o kernel pelo radial, mudaria essa conta?

In [ ]:
from sklearn.pipeline import (
    make_pipeline)
from sklearn.preprocessing \
    import StandardScaler

svc_p = SVC(kernel="linear", C=10)
padron = make_pipeline(
    StandardScaler(), svc_p)
radial = SVC(kernel="rbf", C=10)
candidatos = {
    "padronizado": padron,
    "radial": radial}
for nome, m in candidatos.items():
    m.fit(X_tr, y_tr)
    p_m = m.predict(X_te)
    rotulo = f"erros, {nome}:"
    print(rotulo,
        (p_m != y_te).sum(), "de",
        len(y_te))

Com o mesmo `C` = 10, o kernel linear sobre os genes padronizados também erra 2 das 20 amostras de teste, e o radial, com o `gamma` padrão e os genes sem padronizar, erra 3. Com 20 amostras, uma de diferença não ordena os três ajustes.

Com 4 classes, as formas do `coef_` e do `decision_function` mostram o que o `SVC` ajustou:

In [ ]:
print("coef_:", svc_k.coef_.shape)
d_ovr = svc_k.decision_function(
    X_te)
print("'ovr':", d_ovr.shape)
svc_o = SVC(kernel="linear", C=10,
    decision_function_shape="ovo")
svc_o.fit(X_tr, y_tr)
d_ovo = svc_o.decision_function(
    X_te)
print("'ovo':", d_ovo.shape)
v4 = vota(d_ovo, 4)
i_v = v4.argmax(axis=1)
mais_v = svc_o.classes_[i_v]
print("mais votada = predict:",
    np.all(mais_v == p_te))
print("votos do vencedor:",
    np.unique(v4.max(axis=1)))

O `coef_` tem 6 linhas, uma por par, cada uma com os 2.308 coeficientes de um hiperplano. O `decision_function` tem 6 colunas com `"ovo"` e 4 com `"ovr"`, uma por tumor. Nas 20 amostras de teste, a classe prevista ganha os 3 confrontos de que participa, o máximo possível com 4 classes.

### Quando $p$ é muito maior que $n$

Com 2.308 colunas e 63 amostras, que papel sobra para o `C`? No `SVC`, cada vetor de suporte tem um peso $|\alpha_i|$, o valor absoluto do $\alpha_i$ da [seção 16.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/03-kernels.html), em cada par de que participa. Com $K$ classes, cada vetor de suporte entra nos $K - 1$ pares da sua classe e tem um peso em cada um: o `dual_coef_` tem $K - 1 = 3$ linhas e uma coluna por vetor de suporte. Cada um desses pesos fica entre 0 e o `C`. Um peso zero diz que o ponto não é vetor de suporte daquele par. Um peso entre 0 e o `C`, sem chegar a nenhum dos dois, vem de um ponto sobre a margem daquele par; só um ponto sobre a margem ou do lado errado dela pode chegar ao peso `C`. A tabela conta, para cada `C`, os erros de treino e de teste e os pesos no limite:

In [ ]:
def ajusta(c):
    m = SVC(kernel="linear", C=c)
    m.fit(X_tr, y_tr)
    alfa = np.abs(m.dual_coef_)
    p_tr = m.predict(X_tr)
    p_te = m.predict(X_te)
    er_tr = p_tr != y_tr
    er_te = p_te != y_te
    lim = np.isclose(alfa, c)
    n_lim = lim.sum()
    return m, {"C": c,
        "treino": er_tr.sum(),
        "teste": er_te.sum(),
        "pesos no limite": n_lim}

grade = [0.0001, 0.001, 0.01,
    0.1, 1, 10, 100]
ajustes = [ajusta(c)
    for c in grade]
tab_c = pd.DataFrame(
    [t for _, t in ajustes])
tab_c.style.format(
    {"C": "{:g}"},
    decimal=",").hide()

De 0,01 a 100, o `C` não muda nada: nenhum peso chega ao limite, e os erros são os mesmos. Só os `C` muito pequenos mexem no ajuste. Com `C` = 0,001, 8 pesos batem no limite, e o ajuste ainda não erra nenhuma amostra de treino; com `C` = 0,0001, são 112 pesos, e o ajuste erra 38 das 63 de treino e 14 das 20 de teste. São pesos, não amostras: cada vetor de suporte tem 3, um por par (alguns iguais a zero).

Por que nada muda acima de 0,01? A resposta está no tamanho dos pesos:

In [ ]:
modelos = dict(zip(grade,
    [m for m, _ in ajustes]))
alfa_10 = np.abs(
    modelos[10].dual_coef_)
print("dual_coef_, C = 10:",
    alfa_10.shape)
print("maior |α|, C = 10:",
    numero_br(alfa_10.max(), 3))
ref = modelos[10].coef_
iguais = [np.allclose(
        modelos[c].coef_, ref)
    for c in grade[2:]]
print("coef_ igual, C ≥ 0,01:",
    all(iguais))
a_4 = np.abs(
    modelos[0.0001].dual_coef_)
lim_4 = np.isclose(a_4, 0.0001)
print("C = 0,0001, vetores:",
    lim_4.shape[1])
print("  com peso no limite:",
    lim_4.any(axis=0).sum())

Com `C` = 10, o maior peso é 0,002, abaixo de 0,01. De 0,01 em diante, o limite `C` não alcança nenhum peso, e a solução não muda: os coeficientes são os mesmos de `C` = 0,01 a 100. O `dual_coef_` tem 3 linhas e 54 colunas. Com `C` = 0,0001, os 112 pesos no limite ficam em 59 dos 62 vetores de suporte; `any(axis=0)` marca, em cada coluna, se algum dos 3 pesos chegou ao limite.

Sem nenhum peso no limite, algum ponto de treino viola a margem? Em cada par, $y_i = 1$ para a primeira classe e $-1$ para a segunda, como no sinal das colunas do `"ovo"`:

In [ ]:
d_tr = svc_o.decision_function(
    X_tr)
cl = svc_o.classes_
y_np = y_tr.to_numpy()
yf_min = []
pares_4 = combinations(
    range(4), 2)
for k, (a, b) in enumerate(
        pares_4):
    f = d_tr[:, k]
    yf_a = f[y_np == cl[a]]
    yf_b = -f[y_np == cl[b]]
    yf = np.r_[yf_a, yf_b]
    yf_min.append(yf.min())
print("y·f ≥ 1 − 0,001 nos 6:",
    min(yf_min) >= 1 - 1e-3)

Em cada um dos 6 pares, todo ponto de treino tem $y_i f(x_i)$ de pelo menos 1, com folga de 0,001 (a tolerância do ajuste): está sobre a margem ou além dela, e nenhum a viola. O classificador de cada par é o de margem máxima da [seção 16.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/01-hiperplanos-e-o-classificador-de-margem-maxima.html). Separar as classes de treino, aqui, diz pouco sobre o método: com 2.308 colunas e 63 amostras, os 6 pares deste conjunto se separam por um hiperplano.

Na tabela, o `C` = 0,001 erra uma amostra de teste a menos que os valores de 0,01 a 100. São 20 amostras, e escolher o `C` por esse número seria escolher pelo teste.

### Votos ou escores

E o um contra todos, nos tumores? O `LinearSVC` da [seção 16.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/02-o-classificador-de-vetores-de-suporte.html) usa essa abordagem por padrão, e o `OneVsRestClassifier` a monta com o `SVC`:

In [ ]:
from sklearn.svm import LinearSVC

lin = LinearSVC(C=10,
    random_state=16)
lin.fit(X_tr, y_tr)
print("coef_ do LinearSVC:",
    lin.coef_.shape)
p_lin = lin.predict(X_te)
print("erros, LinearSVC:",
    (p_lin != y_te).sum())
ovr_k = OneVsRestClassifier(
    SVC(kernel="linear", C=10))
ovr_k.fit(X_tr, y_tr)
p_ok = ovr_k.predict(X_te)
print("erros, OneVsRest:",
    (p_ok != y_te).sum())
print("diferentes do SVC:",
    (p_ok != p_te).sum())

O `coef_` do `LinearSVC` tem 4 linhas, uma por tumor, e não 6. Os dois ajustes um contra todos acertam as 20 amostras de teste, e as duas previsões que diferem das do `SVC` são os dois neuroblastomas que ele errou. São 20 amostras, numa só divisão entre treino e teste: não dá para dizer qual abordagem é melhor para esses tumores.

Lado a lado, as duas abordagens:

| | um contra um | um contra todos |
|---|---|---|
| ajustes | $K(K-1)/2$ | $K$ |
| dados de cada um | as duas classes do par | todas as observações |
| decide por | mais votos | maior escore |
| empate | votos iguais (fica a primeira classe) | zero ou vários escores positivos (decide o maior) |

Com 4 tumores, são 6 classificadores contra 4. O número de pares cresce com o quadrado de $K$, e o de classificadores um contra todos, só com $K$. Com $K$ grande, o um contra um faz muitos ajustes pequenos, cada um só com as observações de duas classes, e o um contra todos faz $K$ ajustes, cada um com todas as observações. O `SVC` usa o um contra um, e o `LinearSVC`, o um contra todos. O um contra um cabe quando cada ajuste fica caro com muitas observações, porque cada classificador vê só duas classes; o um contra todos cabe quando $K$ é grande e o número de pares pesa mais que o tamanho de cada ajuste.

## Leituras adicionais

- O capítulo 9 de James et al. (2023), com os exercícios 7, sobre o conjunto `Auto`, e 8, sobre o conjunto `OJ`. O PDF gratuito e os dados estão no [site oficial do livro](https://www.statlearning.com).
- O capítulo 12 de Hastie et al. (2009), "Support Vector Machines and Flexible Discriminants", num tratamento mais técnico do classificador de vetores de suporte e dos kernels.
- O capítulo 5 de Géron (2022), "Support Vector Machines", com os classificadores lineares e com kernel no `scikit-learn`.
- Boser et al. (1992), o artigo do algoritmo de treino dos classificadores de margem ótima.
- Cortes e Vapnik (1995), o artigo que estende a margem máxima a dados de treino que não se separam.
- Chang e Lin (2011), sobre a LIBSVM, a biblioteca que o `SVC` do `scikit-learn` usa.
- Fan et al. (2008), sobre a LIBLINEAR, a biblioteca que o `LinearSVC` usa.
- Khan et al. (2001), o estudo de onde vêm as 83 amostras de tumores da tabela `Khan`.
- [O guia de máquinas de vetores de suporte do scikit-learn](https://scikit-learn.org/stable/modules/svm.html), com o `SVC`, o `LinearSVC`, os kernels e as dicas de escala.

## Referências

- **Boser; Guyon; Vapnik**. *A Training Algorithm for Optimal Margin Classifiers*. ACM. 1992.
- **Chang; Lin**. *LIBSVM: A Library for Support Vector Machines*. 2011.
- **Cortes; Vapnik**. *Support-Vector Networks*. 1995.
- **Fan; Chang; Hsieh; Wang; Lin**. *LIBLINEAR: A Library for Large Linear Classification*. 2008.
- **Géron**. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 3rd ed. O'Reilly Media. 2022.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed. Springer. 2009.
- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.
- **Khan; Wei; Ringnér; Saal; Ladanyi; Westermann; Berthold; Schwab; Antonescu; Peterson; Meltzer**. *Classification and Diagnostic Prediction of Cancers Using Gene Expression Profiling and Artificial Neural Networks*. 2001.